In [1]:
stop

NameError: name 'stop' is not defined

In [3]:
## RUN this BEFORE job submission

import os

_new_directory = "/home/pkwon2/cage" ## change to correct directory

print(f"The current working directory is: {os.getcwd()}")

os.chdir(_new_directory)
print(f"Changed working directory to: {os.getcwd()}")

def create_directory_structure(base_directory):
    try:
        os.makedirs(os.path.join(base_directory, 'input'), exist_ok=True)
        os.makedirs(os.path.join(base_directory, 'outputs'), exist_ok=True)
        print(f"Directory structure '{base_directory}/input' and '{base_directory}/output' created successfully.")
    except Exception as e:
        print(f"Error creating directory structure '{base_directory}': {e}")

if __name__ == "__main__":
    base_directory = "1_diffusion"
    create_directory_structure(base_directory)

The current working directory is: /home/pkwon2/cage
Changed working directory to: /home/pkwon2/cage
Directory structure '1_diffusion/input' and '1_diffusion/output' created successfully.


In [ ]:
#!/usr/bin/env python3

import os

# === User-adjustable settings ===

# Path to the input PDB file
input_pdb = "./input/tile_b_free_diffusion.pdb"

# Number of designs to generate
num_designs = 24

# Numeric contig values to sample (e.g., "150", "160", "170", etc.)
contig_values = ["130"]

# Base prefix for the output directory (no trailing slash; "value1" is literal)
output_prefix_base = "./outputs/c3_tile_b_value1"

# Output directory for the “tasks” file
output_dir = "./1_diffusion/"
os.makedirs(output_dir, exist_ok=True)
tasks_file = os.path.join(output_dir, "tasks")


# === Write out the commands ===

with open(tasks_file, "w") as f:
    for value in contig_values:
        # Build the contigmap.contigs inner string.
        # For chain A: secondary range "218-313"
        # For chains B and C: secondary range "218-303"
        formatted_contigs = (
            f"\"{value}\"-\"{value}\",A\"218\"-\"313\",0\\ "
            f"\"{value}\"-\"{value}\",B\"218\"-\"313\",0\\ "
            f"\"{value}\"-\"{value}\",C\"218\"-\"313\",0"
        )

        # Full output_prefix: ./outputs/c3_tile_b_value1_150
        output_prefix_full = f"{output_prefix_base}_{value}"

        # Construct the barebones command using the example settings
        command = (
            "/software/containers/SE3nv.sif "
            "/net/databases/diffusion/github_repo/rf_diffusion/run_inference.py "
            "--config-name=symmetry "
            "inference.model_runner='NRBStyleSelfCond' "
            "inference.symmetry=\"C3\" "
            f"inference.num_designs={num_designs} "
            f"inference.output_prefix=\"{output_prefix_full}\" "
            "inference.softmax_T=0.0001 "
            "diffuser.T=50 "
            "diffuser.aa_decode_steps=0 "
            "diffuser.chi_type='interp' "
            "potentials.guiding_potentials=[\\'type:olig_contacts,weight_intra:\"0.8\",weight_inter:\"1\"\\'] "
            "potentials.olig_intra_all=True "
            "potentials.olig_inter_all=True "
            "potentials.guide_scale=2 "
            "potentials.guide_decay=\"cubic\" "
            f"contigmap.contigs=[\\'{formatted_contigs}\\'] "
            f"inference.input_pdb=\"{input_pdb}\""
        )

        f.write(command + "\n")

print(f"Tasks written to: {tasks_file}")

In [ ]:
# #!/usr/bin/env python3

# input_pdb = "/mnt/home/pkwon2/cage/1_diffusion/input/T32_top5_5__compA_c3__harley__compB_c2__harley.pdb"
# output_pdb = "/mnt/home/pkwon2/cage/1_diffusion/input/T32_5.pdb"

# allowed_chains = {'A', 'B', 'D', 'E'}

# with open(input_pdb, 'r') as infile, open(output_pdb, 'w') as outfile:
#     for line in infile:
#         if line.startswith(('ATOM', 'HETATM', 'ANISOU', 'TER')):
#             if len(line) >= 22:
#                 chain_id = line[21]
#                 if chain_id in allowed_chains:
#                     outfile.write(line)
#                 else:
#                     # Optionally, write a 'TER' record when changing chains
#                     pass  # You can handle this as needed
#             else:
#                 # Handle lines that are too short (write them as is or skip)
#                 outfile.write(line)
#         else:
#             # Write other lines (e.g., headers, remarks) as is
#             outfile.write(line)

# /software/containers/SE3nv.sif /net/databases/diffusion/github_repo/rf_diffusion/run_inference.py inference.model_runner='NRBStyleSelfCond' inference.num_designs=10 inference.output_prefix="./outputs/T32" inference.softmax_T=0.0001 diffuser.T=50 diffuser.aa_decode_steps=0 diffuser.chi_type='interp' contigmap.contigs=[\'A"1"-"80","5"-"5",C"125"-"143","20"-"30",B"2"-"66",0\ C"69"-"122",0\'] inference.input_pdb="./input/T32_0.pdb" contigmap.inpaint_seq=[\"A73-75\",\"A77-79\",\"C129-143\",\"B2-15\"]

In [4]:
## Using DJ's branch

import itertools
import os

# Define the base command with placeholders
base_command = (
    "/software/containers/SE3nv.sif {script} "
    "--config-name=aa "
    "inference.ckpt_path='{ckpt_path}' "
    "inference.model_runner='NRBStyleSelfCond' "
    "inference.num_designs={num_designs} "
    "inference.output_prefix='{output_prefix}' "
    'contigmap.contigs="{contigs}" '
    "inference.two_template=True "
    "inference.three_template=True "
    "inference.rigid_repeat_motif=False "
    "diffuser.T=50 "
    "model.symmetrize_repeats=False "
    "model.symmsub_k=1 "
    "model.main_block=1 "
    "model.sym_method='max' "
    "denoiser.noise_scale_frame=0.05 "
    "denoiser.noise_scale_ca=0.05 "
    "inference.ij_visible='{ij_visible}' "
    "inference.motif_only_2d=True "
    "preprocess.eye_frames=True "
    "inference.supply_motif_seq=True "
    "inference.cyclic_protein_indices=True "
    "inference.input_pdb='{input_pdb}' "
    "inference.pseudo_symmetry='{n_repeat}' "
    "model.pseudo_cycle=False "
    "inference.align_px0_motif=False"
)

# Script and checkpoint path variables
script = "/projects/small-molecule/rf_diffusion_pseudocycle/rf_diffusion/run_inference.py"
ckpt_paths = [
    "/projects/small-molecule/3template_pseudocycle/BFF_0.20.pt",
]

# Number of designs
num_des = 1  # Adjusted to match your example

# Define the output directory and ensure it exists
output_dir = "./1_diffusion/"
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
tasks_file = os.path.join(output_dir, "tasks")

# Define the pdb_name variable
# pdb_1_name = 'BCDH'
pdb_2_name = 'scratch_af3'
# pdb_3_name = 'BC'
# pdb_4_name = 'partial_tile_a'
# pdb_5_name = 'O43_4'
# pdb_6_name = 'O43_5'

# Example input dictionary with ranges for value1 and value2
input_pdb_dict = {
    # pdb_1_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(91,101))],[list(range(99,115))],[list(range(90,103))],
    #     'contigs_format': "['B2-64,{value1},C20-64,{value2},D9-52,{value3},H20-64,18']",
    #     'num_designs': 1,
    #     'ij_visible': 'abcd',
    #     'input_pdb': f'./input/{pdb_1_name}.pdb',
    #     'n_repeat': 'c1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
    pdb_2_name: {
        # Define ranges for value1 and value2
        'contig_values': [list(range(1,30))],
        'contigs_format': "['A1-63,42-42,B1-63,79-79,C1-16,29-29,D1-13,58-58,E1-63']",
        'num_designs': 2,
        'ij_visible': 'abcde',
        'input_pdb': f'./input/{pdb_2_name}.pdb',
        'n_repeat': 'c1',
        'output_prefix': './outputs/',  # Output directory
    },
    # pdb_3_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(91,101))],
    #     'contigs_format': "['B20-64,91-100,C20-64']",
    #     'num_designs': num_des,
    #     'ij_visible': 'ab',
    #     'input_pdb': f'./input/{pdb_3_name}.pdb',
    #     'n_repeat': 'c1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
    # pdb_4_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(1,31))],
    #     'contigs_format': "['A166-197,35-38,A236-259,35-35,A295-315,52-56,D109-111,17-17,D129-204 C1-51']",
    #     'num_designs': num_des,
    #     'ij_visible': 'abcdef',
    #     'input_pdb': f'./input/{pdb_4_name}.pdb',
    #     'n_repeat': 'c1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
    # pdb_5_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(69,80))],
    #     'contigs_format': "['L16-126,{value1},A451-487 1,F2-3,2,F6-6,3,F10-10,2,F13-13,2']",
    #     'num_designs': num_des,
    #     'ij_visible': 'abcdef',
    #     'input_pdb': f'./input/{pdb_1_name}.pdb',
    #     'n_repeat': 'c1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
    # pdb_6_name: {
    #     # Define ranges for value1 and value2
    #     'contig_values': [list(range(69,80))],
    #     'contigs_format': "['L16-126,{value1},A451-487 1,F2-3,2,F6-6,3,F10-10,2,F13-13,2']",
    #     'num_designs': num_des,
    #     'ij_visible': 'abcdef',
    #     'input_pdb': f'./input/{pdb_1_name}.pdb',
    #     'n_repeat': 'c1',
    #     'output_prefix': './outputs/',  # Output directory
    # },
}

# Function to generate all combinations of values from the provided lists
def generate_combinations(contig_lists):
    return list(itertools.product(*contig_lists))

# Write commands to the file
with open(tasks_file, "w") as file:
    for input_pdb_key, config in input_pdb_dict.items():
        contig_lists = config['contig_values']
        contigs_format = config['contigs_format']
        num_designs = config['num_designs']
        ij_visible = config['ij_visible']
        input_pdb = config['input_pdb']
        n_repeat = config.get('n_repeat', '')
        output_prefix_base = config['output_prefix']
        combinations = generate_combinations(contig_lists)

        for ckpt_path in ckpt_paths:
            for values in combinations:
                # Create a dictionary to hold value placeholders
                value_dict = {f"value{i+1}": str(value) for i, value in enumerate(values)}
                formatted_contigs = contigs_format.format(**value_dict)

                # Create a suffix based on the input PDB, checkpoint, and values
                suffix = "_".join([f"{k}_{v}" for k, v in value_dict.items()])
                suffix = f"{suffix}_ckpt_{os.path.basename(ckpt_path).split('.')[0]}"
                output_prefix_full = os.path.join(output_prefix_base, f"{input_pdb_key}_{suffix}")

                command = base_command.format(
                    script=script,
                    ckpt_path=ckpt_path,
                    num_designs=num_designs,
                    output_prefix=output_prefix_full,
                    contigs=formatted_contigs,
                    ij_visible=ij_visible,
                    input_pdb=input_pdb,
                    n_repeat=n_repeat,
                )

                file.write(command + "\n")

In [6]:
import itertools
import os

# Define the base command with placeholders
base_command = (
    "/software/containers/SE3nv.sif {script} "
    "--config-name=aa "
    "inference.ckpt_path='{ckpt_path}' "
    "inference.model_runner='NRBStyleSelfCond' "
    "inference.num_designs={num_designs} "
    "inference.output_prefix='{output_prefix}' "
    'contigmap.contigs="{contigs}" '
    "inference.two_template=True "
    "inference.three_template=True "
    "inference.rigid_repeat_motif=False "
    "diffuser.T=50 "
    "model.symmetrize_repeats=False "
    "model.symmsub_k=1 "
    "model.main_block=1 "
    "model.sym_method='max' "
    "denoiser.noise_scale_frame=0.05 "
    "denoiser.noise_scale_ca=0.05 "
    "inference.ij_visible='{ij_visible}' "
    "inference.motif_only_2d=True "
    "preprocess.eye_frames=True "
    "inference.supply_motif_seq=True "
    "inference.cyclic_protein_indices=True "
    "inference.input_pdb='{input_pdb}' "
    "inference.pseudo_symmetry='{n_repeat}' "
    "model.pseudo_cycle=False "
    "inference.align_px0_motif=False"
)

# Script and checkpoint path variables
script = "/projects/small-molecule/rf_diffusion_pseudocycle/rf_diffusion/run_inference.py"
ckpt_paths = [
    "/projects/small-molecule/3template_pseudocycle/BFF_0.20.pt",
]

# Number of designs
num_des = 1  # Adjusted to match your example

# Define the output directory and ensure it exists
output_dir = "./1_diffusion/"
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
tasks_file = os.path.join(output_dir, "tasks")

# Define the pdb_name variable
pdb_2_name = 'fiber_to_lhda'

# Example input dictionary with ranges for value1, value2, and value3
input_pdb_dict = {
    pdb_2_name: {
        # Define ranges for value1, value2, and value3 as a list of lists
        'contig_values': [
            list(range(100, 110)),
            list(range(25, 26)),
            list(range(80, 81))
        ],
        # Be sure this format string matches exactly what your tool expects
        'contigs_format': "['A265-356 "
                           "{value1},B95-108,{value2},B125-205,21-21,B227-238,{value3},B265-356"
                           "']",
        'num_designs': num_des,
        'ij_visible': 'abcde',
        'input_pdb': f'./input/{pdb_2_name}.pdb',
        'n_repeat': 'c1',
        'output_prefix': './outputs/',
    },
}

# Function to generate all combinations of values from the provided lists
def generate_combinations(contig_lists):
    return list(itertools.product(*contig_lists))

# Write commands to the file
with open(tasks_file, "w") as file:
    for input_pdb_key, config in input_pdb_dict.items():
        contig_lists     = config['contig_values']
        contigs_format   = config['contigs_format']
        num_designs      = config['num_designs']
        ij_visible       = config['ij_visible']
        input_pdb        = config['input_pdb']
        n_repeat         = config.get('n_repeat', '')
        output_prefix    = config['output_prefix']
        combinations     = generate_combinations(contig_lists)

        for ckpt_path in ckpt_paths:
            for values in combinations:
                # Map each value to value1, value2, ...
                value_dict = {f"value{i+1}": str(v) for i, v in enumerate(values)}
                formatted_contigs = contigs_format.format(**value_dict)

                # Build an output suffix
                suffix = "_".join(f"{k}_{v}" for k, v in value_dict.items())
                suffix = f"{suffix}_ckpt_{os.path.basename(ckpt_path).split('.')[0]}"
                output_prefix_full = os.path.join(output_prefix, f"{input_pdb_key}_{suffix}")

                # Fill in the rest of the placeholders
                command = base_command.format(
                    script=script,
                    ckpt_path=ckpt_path,
                    num_designs=num_designs,
                    output_prefix=output_prefix_full,
                    contigs=formatted_contigs,
                    ij_visible=ij_visible,
                    input_pdb=input_pdb,
                    n_repeat=n_repeat,
                )

                file.write(command + "\n")

In [10]:
# import itertools
# import os

# # Define the new base command with placeholders
# base_command = (
#     "/software/containers/SE3nv.sif /net/databases/diffusion/github_repo/rf_diffusion/run_inference.py "
#     "--config-name=symmetry "
#     "inference.model_runner='NRBStyleSelfCond' "
#     "inference.symmetry=\"C2\" "
#     "inference.num_designs={num_designs} "
#     "inference.output_prefix=\"./outputs/{output_prefix}\" "
#     "inference.softmax_T=0.0001 "
#     "diffuser.T=50 "
#     "diffuser.aa_decode_steps=0 "
#     "diffuser.chi_type='interp' "
#     "potentials.guiding_potentials=[\\'type:olig_contacts,weight_intra:\"0.8\",weight_inter:\"1\"\\'] "
#     "potentials.olig_intra_all=True "
#     "potentials.olig_inter_all=True "
#     "potentials.guide_scale=2 "
#     "potentials.guide_decay=\"cubic\" "
#     "contigmap.contigs='{contigs}' "
#     "inference.input_pdb=\"./input/{input_pdb}.pdb\" "
#     "contigmap.inpaint_seq={inpaint_seq}"
# )

# # Example input dictionary.
# # Note that each entry now includes:
# #  - 'contig_values' and 'contigs_format' (which are used to generate the contigs argument),
# #  - 'num_designs', and a new key 'inpaint_seq'.
# input_pdb_dict = {
#     'c2_tile_b': {
#         'contig_values': [list(range(232, 233))],  # For example, could be used to fill placeholders if needed.
#         'contigs_format': "[\\'\"44\"-\"44\",A\"1\"-\"214\",0\\ \"44\"-\"44\",B\"1\"-\"214\",0\\']",
#         'num_designs': 10,
#     },
#     # Add additional entries here if needed.
# }

# # Ensure the output directory exists
# output_dir = "./1_diffusion/"
# os.makedirs(output_dir, exist_ok=True)

# # Define the output file path
# tasks_file = os.path.join(output_dir, "tasks")

# # Function to generate all combinations of values from the provided lists
# def generate_combinations(contig_lists):
#     return list(itertools.product(*contig_lists)) if contig_lists else [()]

# # Write commands to the file
# with open(tasks_file, "w") as file:
#     for input_pdb, config in input_pdb_dict.items():
#         contig_lists = config.get('contig_values', [])
#         contigs_format = config['contigs_format']
#         num_designs = config['num_designs']
        
#         # Generate combinations if there are multiple values to substitute
#         combinations = generate_combinations(contig_lists)
        
#         for values in combinations:
#             # Create a dictionary to hold placeholders (e.g., {value1}, {value2}, etc.)
#             value_dict = {f"value{i+1}": str(value) for i, value in enumerate(values)}
#             formatted_contigs = contigs_format.format(**value_dict)
            
#             # Create a suffix based on the combination values (if any)
#             suffix = "_".join([f"{k}_{v}" for k, v in value_dict.items()]) if value_dict else ""
#             output_prefix = f"{input_pdb}_{suffix}" if suffix else input_pdb
            
#             # Format the full command with all parameters filled in
#             command = base_command.format(
#                 num_designs=num_designs,
#                 contigs=formatted_contigs,
#                 input_pdb=input_pdb,
#                 output_prefix=output_prefix,
#                 inpaint_seq=inpaint_seq
#             )
#             file.write(command + "\n")

In [8]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p gpu-bf
#SBATCH --mem=32g
#SBATCH -t 00:50:00
#SBATCH -c 4
#SBATCH --gres=gpu
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(output_dir, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{output_dir}\n{submit_command}""")

# Define the SBATCH header, change the email to your email so it doesn't ping me!

#SBATCH -J diffusion
#SBATCH -p gpu
#SBATCH --mem=16g
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH -t 00:50:00
#SBATCH -c 2
#SBATCH --gres=gpu:a4000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

SBATCH script './1_diffusion/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/cage/./1_diffusion/
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [5]:
#!/usr/bin/env python3

import os
import glob

# Set the path to the checkpoint
ckpt_3template = '/projects/ml/aa_template/checkpoints/refine/train_session2023-11-04_1699130567.230787/BFF_3.pt'

# Set the path to the script you want to run
script = '/projects/small-molecule/rf_diffusion_pseudocycle/rf_diffusion/run_inference.py'

# Number of designs
num_designs = 1  # Adjust as needed

# Directory where you want the refine_tasks file
tasks_dir = './1_diffusion/'

# Directory containing the .pdb files relative to tasks_dir
pdb_directory = 'outputs'  # Relative to tasks_dir

# Ensure the tasks directory exists
os.makedirs(tasks_dir, exist_ok=True)

# File to write the commands to
tasks_file = os.path.join(tasks_dir, "refine_tasks")

# Get the list of .pdb files by joining tasks_dir, pdb_directory, and '*.pdb'
pdb_files = glob.glob(os.path.join(tasks_dir, pdb_directory, '*.pdb'))

# Print the last two components of the pdb file paths
for pdb_file in pdb_files:
    print('/'.join(pdb_file.split('/')[-2:]))

# Open the tasks file for writing
with open(tasks_file, 'w') as file:
    # Iterate over each .pdb file
    for pdb_file in pdb_files:
        # Adjust pdb_file to be relative as per your requirement
        relative_pdb_file = '/'.join(['./'] + pdb_file.split('/')[-2:])

        # Construct the command
        command = (
            f"/software/containers/SE3nv.sif {script} "
            "--config-name=aa "
            f"inference.ckpt_path='{ckpt_3template}' "
            "inference.model_runner='NRBStyleSelfCond' "
            f"inference.num_designs={num_designs} "
            "inference.two_template=True "
            "inference.three_template=True "
            "diffuser.T=50 "
            "inference.motif_only_2d=True "
            "inference.supply_motif_seq=True "
            "diffuser.so3_type='random' "
            "diffuser.eucl_type='gaussian' "
            "inference.refine_recycles=4 "
            "model.symmsub_k=1 "
            "inference.refine=True "
            f"inference.input_pdb='{relative_pdb_file}' "
            "model.pseudo_cycle=False "
            "inference.pseudo_symmetry='c1' "
            "model.symmetrize_repeats=False "
            "inference.align_px0_motif=False"
        )

        # Write the command to the tasks file
        file.write(command + "\n")

print(f"Commands have been written to {tasks_file}")

outputs/scratch_af3_value1_20_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_24_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_21_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_25_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_18_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_26_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_8_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_22_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_27_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_19_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_23_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_9_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_25_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_21_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_24_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_20_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_23_ckpt_BFF_0_0.pdb
outputs/scratch_af3_value1_9_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_19_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_27_ckpt_BFF_0_1.pdb
outputs/scratch_af3_value1_8_ckpt_BFF_0_1.pdb
outputs/scratch_a

In [6]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p gpu-bf
#SBATCH --mem=16g
#SBATCH -t 00:05:00
#SBATCH -c 2
#SBATCH --gres=gpu
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the refine_tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" refine_tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(output_dir, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat refine_tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{output_dir}\n{submit_command}""")

SBATCH script './1_diffusion/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/cage/./1_diffusion/
sbatch -a 1-$(cat refine_tasks | wc -l) digs_array_job.sh


In [14]:
import os
import shutil

# Define the paths
output_dir = './1_diffusion/outputs/'
unrefined_dir = os.path.join(output_dir, 'unrefined')

# Create unrefined directory if it doesn't exist
if not os.path.exists(unrefined_dir):
    os.makedirs(unrefined_dir)

# Loop through files in the output directory
for file_name in os.listdir(output_dir):
    if file_name.endswith('.pdb') and 'refined' not in file_name:
        # Construct full file path
        file_path = os.path.join(output_dir, file_name)
        
        # Move file to unrefined directory
        shutil.move(file_path, unrefined_dir)

print("All unrefined .pdb files have been moved to the 'unrefined' directory.")

All unrefined .pdb files have been moved to the 'unrefined' directory.


In [2]:
base_directory = "mpnn"
create_directory_structure(base_directory)

Directory structure 'mpnn/input' and 'mpnn/output' created successfully.


In [ ]:
# import os
# from Bio import PDB

# def find_residues_in_pdb(file_path):
#     parser = PDB.PDBParser(QUIET=True)
#     structure = parser.get_structure('PDB_structure', file_path)
#     residue_positions = []

#     for model in structure:
#         for chain in model:
#             for residue in chain:
#                 if residue.get_resname() == 'GLY':
#                     residue_positions.append(f"{chain.id}{residue.id[1]}")

#     return ' '.join(residue_positions)

# def main():
#     pdb_dir = './mpnn/input'
#     pdb_dir = os.path.abspath(pdb_dir)

#     if not os.path.exists(pdb_dir):
#         print(f"Directory {pdb_dir} does not exist.")
#         return

#     base_directory = "mpnn"
#     os.makedirs(base_directory, exist_ok=True)

#     # Define the base command with correct formatting keys
#     base_command = (
#         "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
#         "--redesigned_residues '{residue_positions}' "
#         "--homo_oligomer 1 "
#         "--pdb_path {file_path} "
#         "--out_folder outputs "
#         "--batch_size 2 "
#         "--number_of_batches 2 "
#         "--bias_AA 'M:-10.0' "
#         "--bias_AA 'R:-0.9' "
#         "--pack_side_chains 1 "
#         "--repack_everything 1"
#     )

#     tasks_file = os.path.join(base_directory, "tasks")

#     with open(tasks_file, "w") as file:
#         for pdb_file in os.listdir(pdb_dir):
#             if pdb_file.endswith('.pdb'):
#                 file_path = os.path.join(pdb_dir, pdb_file)
#                 residues = find_residues_in_pdb(file_path)
#                 command = base_command.format(file_path=file_path, residue_positions=residues)
#                 file.write(f"{command}\n")

#     print(f"Tasks file '{tasks_file}' has been created.")

# if __name__ == "__main__":
#     main()

In [ ]:
import os
from Bio import PDB

def find_non_charged_residues(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    residue_positions = []
    
    # Define the list of charged residues
    charged_residues = ['GLU']

    for model in structure:
        for chain in model:
            # Get all residues in the chain as a list
            residues_list = list(chain.get_residues())
            # Exclude the last 88 residues from processing
            if len(residues_list) > 1:
                residues_to_process = residues_list[:]
            else:
                residues_to_process = []  # Skip if the chain is too short

            for residue in residues_to_process:
                resname = residue.get_resname()
                residue_number = residue.id[1]
                # Exclude residues in positions 1-27 and 35-81
                # if 1 <= residue_number <= 27 or 35 <= residue_number <= 81:
                #     continue
                # Add if the residue is not charged
                if resname not in charged_residues:
                    residue_positions.append(f"{chain.id}{residue_number}")
                    
    return ' '.join(residue_positions)

def main():
    pdb_dir = './1_diffusion/outputs'
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Base directory for output tasks
    base_directory = "mpnn"
    os.makedirs(base_directory, exist_ok=True)

    # Define the base command with updated placeholder {residues} can add back                    "--model_type 'ligand_mpnn' " \
    base_command = "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py " \
                   "--model_type 'ligand_mpnn' " \
                   "--temperature 0.3 " \
                   "--redesigned_residues '{residues}' " \
                   "--pdb_path {file_path} " \
                   "--out_folder outputs " \
                   "--batch_size 2 " \
                   "--number_of_batches 2 " \
                   "--bias_AA 'M:-10.0' " \
                   "--bias_AA 'R:-0.9' " \
                   "--pack_side_chains 1 " \
                   "--repack_everything 1"

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                residues = find_non_charged_residues(file_path)
                command = base_command.format(file_path=file_path, residues=residues)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [ ]:
# import os
# from Bio import PDB

# def find_residues_in_pdb(file_path):
#     parser = PDB.PDBParser(QUIET=True)
#     structure = parser.get_structure('PDB_structure', file_path)
#     residue_positions = []

#     for model in structure:
#         for chain in model:
#             for residue in chain:
#                 if residue.get_resname() in ['PHE']:
#                     residue_positions.append(f"{chain.id}{residue.id[1]}")

#     return ' '.join(residue_positions)

# def main():
#     pdb_dir = './mpnn/input'
    
#     # Convert relative path to absolute path
#     pdb_dir = os.path.abspath(pdb_dir)

#     if not os.path.exists(pdb_dir):
#         print(f"Directory {pdb_dir} does not exist.")
#         return
#     # Base directory for output tasks
#     base_directory = "mpnn"
#     os.makedirs(base_directory, exist_ok=True)

#     # Define the base command
#     base_command = "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py " \
#                    "--temperature 0.3 " \
#                    "--redesigned_residues '{residue_positions}' " \
#                    "--homo_oligomer 1 " \
#                    "--pdb_path {file_path} " \
#                    "--out_folder outputs " \
#                    "--batch_size 5 " \
#                    "--number_of_batches 5 " \
#                    "--bias_AA 'M:-10.0' " \
#                    "--bias_AA 'R:-0.9' " \
#                    "--pack_side_chains 1 " \
#                    "--repack_everything 1"

#     tasks_file = os.path.join(base_directory, "tasks")

#     with open(tasks_file, "w") as file:
#         for pdb_file in os.listdir(pdb_dir):
#             if pdb_file.endswith('.pdb'):
#                 file_path = os.path.join(pdb_dir, pdb_file)
#                 residues = find_residues_in_pdb(file_path)
#                 command = base_command.format(file_path=file_path, residues=residues)
#                 file.write(f"{command}\n")

#     print(f"Tasks file '{tasks_file}' has been created.")

# if __name__ == "__main__":
#     main()

## Second round of MPNN

import os
from Bio import PDB

def find_non_charged_residues(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    residue_positions = []
    
    # Define the list of charged residues
    charged_residues = ['ASP', 'GLU', 'SER']

    for model in structure:
        for chain in model:
            # Get all residues in the chain as a list
            residues_list = list(chain.get_residues())
            # Exclude the last 88 residues from processing
            if len(residues_list) > 88:
                residues_to_process = residues_list[:-88]
            else:
                residues_to_process = []  # Skip if the chain is too short

            for residue in residues_to_process:
                resname = residue.get_resname()
                residue_number = residue.id[1]
                # Exclude residues in positions 1-27 and 35-81
                # if 1 <= residue_number <= 27 or 35 <= residue_number <= 81:
                #     continue
                # Add if the residue is not charged
                if resname not in charged_residues:
                    residue_positions.append(f"{chain.id}{residue_number}")
                    
    return ' '.join(residue_positions)

def main():
    pdb_dir = './mpnn/input'
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Base directory for output tasks
    base_directory = "mpnn"
    os.makedirs(base_directory, exist_ok=True)

    # Define the base command with updated placeholder {residues}
    base_command = "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py " \
                   "--temperature 0.3 " \
                   "--redesigned_residues '{residues}' " \
                   "--homo_oligomer 1 " \
                   "--pdb_path {file_path} " \
                   "--out_folder outputs " \
                   "--batch_size 4 " \
                   "--number_of_batches 4 " \
                   "--bias_AA 'M:-10.0' " \
                   "--pack_side_chains 1 " \
                   "--repack_everything 1"

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                residues = find_non_charged_residues(file_path)
                command = base_command.format(file_path=file_path, residues=residues)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [ ]:
import os

def build_symmetry_groups(chains, ranges):
    """
    Given a list of chain IDs and a list of (start, end) tuples,
    return a list of symmetry‐residue group strings like ["A1,B1,…,I1", …].
    """
    groups = []
    for start, end in ranges:
        for resnum in range(start, end + 1):
            labels = [f"{chain}{resnum}" for chain in chains]
            groups.append(",".join(labels))
    return groups

def main():
    # define your chain sets and their residue‐ranges
    group1_chains = list("ABCDEFGHI")
    group1_ranges = [(1, 29), (34, 50), (52, 81)]

    group2_chains = list("DEFGHI")
    group2_ranges = [(30, 33), (82, 101)]

    group3_chains = list("ABC")
    group3_ranges = [(30, 33), (82, 285), (288, 289), (292, 293), (295, 297), (299, 300), (303, 304), (307, 337), (340, 341), (344, 345), (347, 349), (351, 352), (355, 356), (359, 389), (392, 393), (396, 397), (399, 401), (403, 404), (407, 408), (411, 468)]

    # build all symmetry groups in order
    all_groups = (
        build_symmetry_groups(group1_chains, group1_ranges) +
        build_symmetry_groups(group2_chains, group2_ranges) +
        build_symmetry_groups(group3_chains, group3_ranges)
    )

    # flatten every individual residue for redesigned_residues, separated by spaces
    redesigned_residues = " ".join(
        res for group in all_groups for res in group.split(",")
    )

    # join for the symmetry flags
    symmetry_residues = "|".join(all_groups)
    symmetry_weights  = "|".join(
        ",".join(["1"] * len(group.split(",")))
        for group in all_groups
    )

    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--temperature 0.3 "
        "--homo_oligomer 1 "
        "--pdb_path {file_path} "
        "--out_folder outputs "
        "--batch_size 10 "
        "--number_of_batches 10 "
        "--bias_AA 'M:-10.0' "
        "--bias_AA 'R:-0.9' "
        "--pack_side_chains 1 "
        "--repack_everything 1 "
        f"--redesigned_residues '{redesigned_residues}' "
        f"--symmetry_residues '{symmetry_residues}' "
        f"--symmetry_weights '{symmetry_weights}'"
    )

    input_dir = os.path.abspath("./mpnn/input")
    os.makedirs("mpnn", exist_ok=True)
    tasks_file = os.path.join("mpnn", "tasks")

    with open(tasks_file, "w") as out:
        for fname in os.listdir(input_dir):
            if not fname.endswith(".pdb"):
                continue
            path = os.path.join(input_dir, fname)
            cmd = base_command.format(file_path=path)
            out.write(cmd + "\n")

    print(f"Tasks file written to {tasks_file}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import re
import numpy as np

def get_trb_file_for_pdb(pdb_file):
    """
    Given a pdb_file name, remove the '_refined_X.pdb' suffix if present
    and return the corresponding TRB file name.
    """
    # Remove a refined suffix like _refined_1.pdb, _refined_2.pdb, etc.
    base_pdb = re.sub(r'_refined_\d+\.pdb$', '.pdb', pdb_file)
    trb_file = base_pdb.replace('.pdb', '.trb')
    return trb_file

def get_redesigned_residues_from_trb(pdb_file):
    """
    For the given pdb_file, this function locates the corresponding TRB file
    by first removing any '_refined_X.pdb' suffix. It then loads the TRB data (assumed
    to contain a key 'mask_1d' which is a boolean array), prints out the indexes where
    the mask is False, and uses the mask to determine which unique residues (based on
    the first ATOM line for each residue) need to be redesigned.
    
    Redesigned residues are enumerated as the chain letter followed by (unique residue index + 1).
    For example, if the false indexes are [27, 28, 29, 30] on chain A, the labels will be:
    A28 A29 A30 A31.
    
    Returns a space-separated string of these residue labels.
    """
    # Get the base TRB file name from the pdb file.
    trb_file = get_trb_file_for_pdb(pdb_file)
    if not os.path.exists(trb_file):
        print(f"TRB file not found for {pdb_file} (expected {trb_file})")
        return ""
    
    # Load TRB data (assumed to have key 'mask_1d' as an array of booleans)
    trb_data = np.load(trb_file, allow_pickle=True)
    mask = trb_data['mask_1d']
    
    # Print the indexes where mask_1d is False.
    false_indices = [i for i, value in enumerate(mask) if not value]
    print(f"For file '{pdb_file}', mask_1d False indices: {false_indices}")
    
    redesigned_residues = []
    
    # Collect unique residues (first occurrence per residue) from the pdb file.
    seen = set()
    unique_residues = []  # List of tuples: (chain, resi)
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                parts = line.split()
                # Assuming PDB columns: index 4 is chain and index 5 is residue number.
                chain = parts[4]
                resi = parts[5]
                if (chain, resi) not in seen:
                    seen.add((chain, resi))
                    unique_residues.append((chain, resi))
    
    # Enumerate through unique residues and label those with mask False.
    for i, (chain, resi) in enumerate(unique_residues):
        if i < len(mask) and not mask[i]:
            # Label is the chain letter followed by (index + 1)
            redesigned_residues.append(f"{chain}{i+1}")
    
    return " ".join(redesigned_residues)

def main():
    pdb_dir = './1_diffusion/outputs'
    
    # Convert relative path to absolute path.
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Base directory for output tasks.
    base_directory = "mpnn"
    os.makedirs(base_directory, exist_ok=True)

    # Define the base command with the redesigned residues placeholder.
    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--temperature 0.3 "
        "--redesigned_residues '{residues}' "
        "--pdb_path {file_path} "
        "--out_folder outputs "
        "--batch_size 2 "
        "--number_of_batches 2 "
        "--bias_AA 'M:-10.0' "
        "--bias_AA 'R:-0.9' "
        "--pack_side_chains 1 "
        "--repack_everything 1"
    )

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                # Get the residues to be redesigned from the corresponding TRB file.
                residues = get_redesigned_residues_from_trb(file_path)
                # Format the command with the pdb path and residues string.
                command = base_command.format(
                    file_path=file_path,
                    residues=residues
                )
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import re
import numpy as np

def get_trb_file_for_pdb(pdb_file):
    base_pdb = re.sub(r'_refined_\d+\.pdb$', '.pdb', pdb_file)
    trb_file = base_pdb.replace('.pdb', '.trb')
    return trb_file

def get_redesigned_residues_from_trb(pdb_file):
    trb_file = get_trb_file_for_pdb(pdb_file)
    if not os.path.exists(trb_file):
        print(f"TRB file not found for {pdb_file} (expected {trb_file})")
        return ""

    trb_data = np.load(trb_file, allow_pickle=True)
    mask = np.asarray(trb_data['mask_1d'], dtype=bool)
    inpaint_seq = np.asarray(trb_data.get('inpaint_seq', np.ones_like(mask, dtype=bool)), dtype=bool)

    # Collect indices where either mask or inpaint_seq is False
    false_indices = set(np.where(~mask)[0]) | set(np.where(~inpaint_seq)[0])

    # Print indices (1-based)
    print(f"For file '{pdb_file}', combined False indices: {[idx+1 for idx in sorted(false_indices)]}")

    # Read pdb file and extract unique residues with corresponding chains
    unique_residues = []
    seen_residues = set()
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                chain = line[21].strip()
                resi = line[22:26].strip()
                residue_key = (chain, resi)
                if residue_key not in seen_residues:
                    seen_residues.add(residue_key)
                    unique_residues.append(residue_key)

    # Generate redesigned residue labels based on false indices
    redesigned_residues = []
    for idx in sorted(false_indices):
        if idx < len(unique_residues):
            chain, resi = unique_residues[idx]
            label = f"{chain}{resi}"
            redesigned_residues.append(label)
        else:
            print(f"Warning: index {idx} out of range for residues in {pdb_file}")

    return " ".join(redesigned_residues)

def main():
    pdb_dir = './1_diffusion/outputs'
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    base_directory = "mpnn"
    os.makedirs(base_directory, exist_ok=True)

    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--temperature 0.3 "
        "--redesigned_residues '{residues}' "
        "--pdb_path {file_path} "
        "--out_folder outputs "
        "--batch_size 2 "
        "--number_of_batches 2 "
        "--bias_AA 'M:-10.0' "
        "--bias_AA 'R:-0.9' "
        "--pack_side_chains 1 "
        "--repack_everything 1"
    )

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                residues = get_redesigned_residues_from_trb(file_path)
                command = base_command.format(
                    file_path=file_path,
                    residues=residues
                )
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [3]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J mpnn
#SBATCH -p cpu
#SBATCH --cpus-per-task=4
#SBATCH -t 01:00:00
#SBATCH --mem=48g
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

SBATCH script 'mpnn/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/cage/mpnn
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [5]:
import os

def keep_chain_a(input_file, output_file):
    """
    Keeps only chain A in a PDB file and writes the result to the output file.
    """
    with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
        for line in f_in:
            # Check if the line corresponds to an ATOM or HETATM record and chain is A
            if (line.startswith("ATOM") or line.startswith("HETATM")) and line[21] == 'B':
                f_out.write(line)
            # Write other header/footer lines (e.g., TER, END, etc.)
            elif not (line.startswith("ATOM") or line.startswith("HETATM")):
                f_out.write(line)

def process_pdb_files(input_directory, output_directory):
    """
    Processes all .pdb files in the specified directory, keeping only chain A.
    Writes the modified files to the output directory.
    """
    # Create output directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    for filename in os.listdir(input_directory):
        if filename.endswith(".pdb"):
            input_file = os.path.join(input_directory, filename)
            output_file = os.path.join(output_directory, filename)
            keep_chain_a(input_file, output_file)
            print(f"Processed {filename}")

if __name__ == "__main__":
    input_directory = './mpnn/outputs/packed'
    output_directory = './mpnn/outputs/alphafold'
    
    process_pdb_files(input_directory, output_directory)

Processed fiber_to_lhda_value1_100_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_15_1.pdb
Processed fiber_to_lhda_value1_106_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_14_1.pdb
Processed fiber_to_lhda_value1_104_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_16_1.pdb
Processed fiber_to_lhda_value1_108_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_13_1.pdb
Processed fiber_to_lhda_value1_104_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_6_1.pdb
Processed fiber_to_lhda_value1_100_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_4_1.pdb
Processed fiber_to_lhda_value1_108_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_9_1.pdb
Processed fiber_to_lhda_value1_101_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_6_1.pdb
Processed fiber_to_lhda_value1_108_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_2_1.pdb
Processed fiber_to_lhda_value1_105_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_4_1.pdb
Processed fiber_to_lhda_value1_107_value2_25_value3_80_ckpt_BFF_0_

In [6]:
base_directory = "superfold"
create_directory_structure(base_directory)

def main():
    pdb_dir = './mpnn/outputs/alphafold' #this is where your mpnn outputs are
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Define the base command
    base_command = "/net/software/lab/superfold/pyroml/bin/python3.8 /software/lab/superfold/run_superfold.py {file_path} " \
                   "--models 3 " \
                   "--initial_guess " \
                   "--out_dir outputs " \

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                command = base_command.format(file_path=file_path)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

Directory structure 'superfold/input' and 'superfold/output' created successfully.
Tasks file 'superfold/tasks' has been created.


In [1]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J af2
#SBATCH -p cpu
#SBATCH --mem=32g
#SBATCH -t 00:30:00
#SBATCH -c 2
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

NameError: name 'os' is not defined

In [10]:
base_directory = "superfold"
create_directory_structure(base_directory)

import os
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser, Superimposer
import shutil

# Define directories
packed_dir = './mpnn/outputs/alphafold'
superfold_outputs_dir = './superfold/outputs'
rmsd_filtered_dir = './superfold/rmsd_filtered'

# Create the filtered directory if it doesn't exist
os.makedirs(rmsd_filtered_dir, exist_ok=True)

# Initialize PDB parser
parser = PDBParser(QUIET=True)

def calculate_rmsd(file1, file2):
    """Calculate RMSD between two PDB files using CA atoms."""
    try:
        structure1 = parser.get_structure('structure1', file1)
        structure2 = parser.get_structure('structure2', file2)

        # Extract CA atoms and order by residue number
        atoms1 = []
        atoms2 = []
        for model1, model2 in zip(structure1, structure2):
            for chain1, chain2 in zip(model1, model2):
                residues1 = [res for res in chain1 if res.has_id('CA')]
                residues2 = [res for res in chain2 if res.has_id('CA')]

                if len(residues1) != len(residues2):
                    print(f'Number of CA residues differ between {file1} and {file2}')
                    return None

                for res1, res2 in zip(residues1, residues2):
                    atoms1.append(res1['CA'])
                    atoms2.append(res2['CA'])

        if len(atoms1) != len(atoms2):
            print(f'Number of CA atoms differ between {file1} and {file2}')
            return None

        # Superimpose structures
        super_imposer = Superimposer()
        super_imposer.set_atoms(atoms1, atoms2)
        super_imposer.apply(structure2.get_atoms())

        return super_imposer.rms
    except Exception as e:
        print(f'Error calculating RMSD between {file1} and {file2}: {e}')
        return None

# Dataframe to store paths and RMSD values
data = []

# Iterate over files in the packed directory
for packed_pdb in os.listdir(packed_dir):
    if packed_pdb.endswith('.pdb'):
        packed_pdb_path = os.path.join(packed_dir, packed_pdb)
        # Loop over models 1 to 5
        for model_num in range(3, 4):
            # Construct corresponding file names in superfold directory
            superfold_pdb = packed_pdb.replace('.pdb', f'_model_{model_num}_ptm_seed_0_unrelaxed.pdb')
            print(f'Processing: {superfold_pdb}')
            superfold_json = superfold_pdb.replace('unrelaxed.pdb', 'prediction_results.json')
            
            superfold_pdb_path = os.path.join(superfold_outputs_dir, superfold_pdb)
            superfold_json_path = os.path.join(superfold_outputs_dir, superfold_json)
            
            # Ensure the corresponding superfold pdb and json files exist
            if os.path.exists(superfold_pdb_path) and os.path.exists(superfold_json_path):
                # Calculate RMSD
                rmsd = calculate_rmsd(packed_pdb_path, superfold_pdb_path)
                
                if rmsd is not None:
                    # Check RMSD threshold
                    if rmsd < 1.5:
                        # Copy pdb and json files to rmsd_filtered directory
                        shutil.copy(superfold_pdb_path, rmsd_filtered_dir)
                        shutil.copy(superfold_json_path, rmsd_filtered_dir)
                        
                        # Add to dataframe
                        data.append([packed_pdb_path, superfold_pdb_path, rmsd])
                else:
                    print(f'Skipping due to RMSD calculation error between {packed_pdb_path} and {superfold_pdb_path}')
            else:
                print(f'Missing files for {superfold_pdb_path} or {superfold_json_path}')

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data, columns=['Packed PDB Path', 'Superfold PDB Path', 'RMSD'])
csv_path = os.path.join(rmsd_filtered_dir, 'rmsd_filtered.csv')
df.to_csv(csv_path, index=False)

print(f'RMSD calculation and file filtering complete. Results saved to {csv_path}')

Directory structure 'superfold/input' and 'superfold/output' created successfully.
Processing: fiber_to_lhda_value1_102_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_16_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_106_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_15_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_108_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_6_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_101_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_104_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_9_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_104_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_2_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_101_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_9_1_model_3_ptm_seed_0_unrelaxed.pdb
Processing: fiber_to_lhda_value1_109_value2_25_

In [ ]:
# import os
# import pandas as pd
# import numpy as np
# from Bio.PDB import PDBParser
# import shutil

# # Define directories
# packed_dir = './mpnn/outputs/alphafold'
# superfold_outputs_dir = './superfold/outputs'
# rmsd_filtered_dir = './superfold/rmsd_filtered'

# # Create the filtered directory if it doesn't exist
# os.makedirs(rmsd_filtered_dir, exist_ok=True)

# # Initialize PDB parser
# parser = PDBParser()

# def kabsch(P, Q):
#     """
#     Compute the optimal rotation matrix using the Kabsch algorithm.
#     P and Q are (N, 3) numpy arrays of corresponding points.
#     """
#     # Subtract centroids
#     centroid_P = np.mean(P, axis=0)
#     centroid_Q = np.mean(Q, axis=0)
#     P_centered = P - centroid_P
#     Q_centered = Q - centroid_Q

#     # Compute covariance matrix
#     H = np.dot(Q_centered.T, P_centered)
#     U, S, Vt = np.linalg.svd(H)
#     R = np.dot(Vt.T, U.T)
#     # Ensure a right-handed coordinate system
#     if np.linalg.det(R) < 0:
#         Vt[-1, :] *= -1
#         R = np.dot(Vt.T, U.T)
#     T = centroid_P - np.dot(R, centroid_Q)
#     return R, T

# def calculate_rmsd_icp(file1, file2, max_iterations=100, tolerance=1e-5):
#     """
#     Calculate RMSD between two PDB files using a simple ICP algorithm.
#     This does not require the two structures to have the same number of atoms.
#     For consistency, we use only CA atoms.
#     """
#     structure1 = parser.get_structure('structure1', file1)
#     structure2 = parser.get_structure('structure2', file2)
    
#     # Extract only CA atoms
#     fixed_atoms = np.array([atom.get_coord() for atom in structure1.get_atoms() if atom.get_name() == 'CA'])
#     moving_atoms = np.array([atom.get_coord() for atom in structure2.get_atoms() if atom.get_name() == 'CA'])
    
#     if fixed_atoms.shape[0] == 0 or moving_atoms.shape[0] == 0:
#         raise Exception("No CA atoms found in one of the structures.")
    
#     # Start with the original moving atoms
#     transformed_moving = moving_atoms.copy()
#     prev_rmsd = None

#     for i in range(max_iterations):
#         # For each point in the moving structure, find the closest point in the fixed structure.
#         # Compute distance matrix between each point in transformed_moving and fixed_atoms.
#         diff = transformed_moving[:, np.newaxis, :] - fixed_atoms[np.newaxis, :, :]
#         distances = np.linalg.norm(diff, axis=2)
#         nearest_indices = distances.argmin(axis=1)
#         matched_fixed = fixed_atoms[nearest_indices]
        
#         # Compute optimal rotation and translation that align transformed_moving to matched_fixed.
#         R, T = kabsch(matched_fixed, transformed_moving)
        
#         # Apply transformation to moving atoms.
#         transformed_moving = (np.dot(R, transformed_moving.T).T) + T
        
#         # Calculate RMSD over the matched pairs.
#         diff_after = transformed_moving - matched_fixed
#         rmsd = np.sqrt((diff_after**2).sum() / len(diff_after))
        
#         if prev_rmsd is not None and abs(prev_rmsd - rmsd) < tolerance:
#             break
#         prev_rmsd = rmsd

#     return rmsd

# # Dataframe to store paths and RMSD values
# data = []

# # Iterate over files in the packed directory
# for packed_pdb in os.listdir(packed_dir):
#     if packed_pdb.endswith('.pdb'):
#         # Construct corresponding file names in superfold directory
#         superfold_pdb = packed_pdb.replace('.pdb', '_b_model.pdb')
#         print(superfold_pdb)
#         superfold_json = superfold_pdb.replace('unrelaxed.pdb', 'prediction_results.json')
        
#         superfold_pdb_path = os.path.join(superfold_outputs_dir, superfold_pdb)
#         superfold_json_path = os.path.join(superfold_outputs_dir, superfold_json)
        
#         # Ensure the corresponding superfold pdb and json files exist
#         if os.path.exists(superfold_pdb_path) and os.path.exists(superfold_json_path):
#             packed_pdb_path = os.path.join(packed_dir, packed_pdb)
            
#             # Calculate RMSD using the ICP-based algorithm
#             rmsd = calculate_rmsd_icp(packed_pdb_path, superfold_pdb_path)
            
#             # Check RMSD threshold
#             if rmsd < 20:
#                 # Copy pdb and json files to rmsd_filtered directory
#                 shutil.copy(superfold_pdb_path, rmsd_filtered_dir)
#                 shutil.copy(superfold_json_path, rmsd_filtered_dir)
                
#                 # Add to dataframe
#                 data.append([packed_pdb_path, superfold_pdb_path, rmsd])

# # Convert to DataFrame and save to CSV
# df = pd.DataFrame(data, columns=['Packed PDB Path', 'Superfold PDB Path', 'RMSD'])
# csv_path = os.path.join(rmsd_filtered_dir, 'rmsd_filtered.csv')
# df.to_csv(csv_path, index=False)

# print(f'RMSD calculation and file filtering complete. Results saved to {csv_path}')

In [11]:
import os
import json

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Loop through all files in the directory
for filename in os.listdir(directory):
    # Check if the file is a .json file
    if filename.endswith('prediction_results.json'):
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_plddt value
            mean_plddt = data.get('mean_plddt', 1.0)  # Default to 1.0 if not found
            
            # Print the mean_plddt value and the .json filename
            # print(f'{filename}: mean_plddt = {mean_plddt}')
            
            # Check if mean_plddt is less than 80
            if mean_plddt < 90:
                # Construct the corresponding .pdb filename
                pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
                # print(pdb_filename)
                pdb_path = os.path.join(directory, pdb_filename)
                # print(pdb_path)
                # Delete the .pdb file if it exists
                if os.path.exists(pdb_path):
                    os.remove(pdb_path)
                    print(f'Deleted {pdb_path}')
                else:
                    print(f'{pdb_path} does not exist')
                
                # Delete the .json file
                os.remove(json_path)
                print(f'Deleted {json_path}')

print('Processing complete.')

Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_103_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_16_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_103_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_16_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_109_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_14_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_109_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_14_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_102_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_6_1_model_3_ptm_seed_0_unrelaxed.pdb
Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_102_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_6_1_model_3_ptm_seed_0_prediction_results.json
Deleted ./superfold/rmsd_filtered/fiber_to_lhda_value1_109_value2_25_value3_80_ckpt_BFF_0_

In [ ]:
# import os
# import json
# from collections import defaultdict

# # Define the directory containing the .json and .pdb files
# directory = './superfold/rmsd_filtered'

# # Initialize a dictionary to hold groups of files
# file_groups = defaultdict(list)

# # Function to extract the prefix based on the last instance of 'packed'
# def get_prefix(filename):
#     parts = filename.split('packed')
#     if len(parts) > 1:
#         return 'packed'.join(parts[:-1]) + 'packed'
#     return filename

# # Loop through all files in the directory
# for filename in os.listdir(directory):
#     if filename.endswith('prediction_results.json'):
#         prefix = get_prefix(filename)
#         json_path = os.path.join(directory, filename)
        
#         # Read the .json file
#         with open(json_path, 'r') as json_file:
#             data = json.load(json_file)
            
#             # Get the mean_plddt value
#             mean_plddt = data.get('mean_plddt', 0.0)  # Default to 0.0 if not found
            
#             # Append the file info to the group
#             file_groups[prefix].append((mean_plddt, json_path, filename))

# # Function to delete a file
# def delete_file(file_path):
#     if os.path.exists(file_path):
#         os.remove(file_path)
#         print(f'Deleted {file_path}')
#     else:
#         print(f'{file_path} does not exist')

# # Iterate through each group
# for prefix, files in file_groups.items():
#     # Sort the files by mean_plddt in descending order
#     files.sort(reverse=True, key=lambda x: x[0])
    
#     # Get the highest n files (modify 'n' to choose highest 1, 2, 3, etc.)
#     n = 1  # For highest 1 file per group, change to 3 for highest 3 per group
#     highest_files = files[:n]
    
#     # Files to keep
#     highest_files_set = set(f[1] for f in highest_files)
    
#     # Iterate through the files in the group
#     for _, json_path, filename in files:
#         if json_path not in highest_files_set:
#             # Delete the .json file
#             delete_file(json_path)
            
#             # Construct the corresponding .pdb filename and delete it
#             pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
#             pdb_path = os.path.join(directory, pdb_filename)
#             delete_file(pdb_path)

# print('Processing complete.')

import os
import json
from collections import defaultdict

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Initialize a dictionary to hold groups of files
file_groups = defaultdict(list)

# Function to extract the prefix based on the last instance of 'packed'
def get_prefix(filename):
    parts = filename.split('packed')
    if len(parts) > 1:
        return 'packed'.join(parts[:-1]) + 'packed'
    return filename

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('prediction_results.json'):
        prefix = get_prefix(filename)
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_pae value; default to a high value if not found so it doesn't win
            mean_pae = data.get('mean_pae', 999)
            
            # Append the file info to the group
            file_groups[prefix].append((mean_pae, json_path, filename))

# Function to delete a file
def delete_file(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted {file_path}')
    else:
        print(f'{file_path} does not exist')

# Iterate through each group
for prefix, files in file_groups.items():
    # Sort the files by mean_pae in ascending order (lowest value first)
    files.sort(key=lambda x: x[0])
    
    # Get the lowest n files (modify 'n' to choose lowest 1, 2, 3, etc.)
    n = 1 # For the lowest 1 file per group; change to 3 for the lowest 3 per group
    lowest_files = files[:n]
    
    # Files to keep
    lowest_files_set = set(f[1] for f in lowest_files)
    
    # Iterate through the files in the group and delete those not selected
    for _, json_path, filename in files:
        if json_path not in lowest_files_set:
            # Delete the .json file
            delete_file(json_path)
            
            # Construct the corresponding .pdb filename and delete it
            pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
            pdb_path = os.path.join(directory, pdb_filename)
            delete_file(pdb_path)

print('Processing complete.')

In [13]:
import os
import shutil
from glob import glob

# Paths
superfold_dir = './superfold/rmsd_filtered/'
mpnn_dir = './mpnn/outputs/packed/'
output_dir = './mpnn/outputs/first_pass_superfold/'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get list of PDB files in superfold directory
superfold_pdbs = glob(os.path.join(superfold_dir, '*.pdb'))

# Iterate through each PDB in superfold
for superfold_pdb in superfold_pdbs:
    # Get the base name of the PDB (without directory)
    superfold_filename = os.path.basename(superfold_pdb)
    
    # Extract the prefix before '_model'
    prefix = superfold_filename.split('_model')[0]
    
    # Search for corresponding files in the mpnn directory
    mpnn_pdb_pattern = os.path.join(mpnn_dir, f'{prefix}*.pdb')
    mpnn_pdbs = glob(mpnn_pdb_pattern)
    
    # If matching files are found, copy them to the new directory
    for mpnn_pdb in mpnn_pdbs:
        shutil.copy(mpnn_pdb, output_dir)

print(f"Copying completed. Files are copied to {output_dir}.")

Copying completed. Files are copied to ./mpnn/outputs/first_pass_superfold/.


In [ ]:
base_directory = "superfold"
create_directory_structure(base_directory)

def main():
    pdb_dir = './mpnn/outputs/first_pass_superfold' #this is where your mpnn outputs are
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Define the base command
    base_command = "/net/software/lab/superfold/pyroml/bin/python3.8 /software/lab/superfold/run_superfold.py {file_path} " \
                   "--initial_guess " \
                   "--out_dir outputs " \

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                command = base_command.format(file_path=file_path)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [ ]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J af2
#SBATCH -p gpu
#SBATCH --mem=16g
#SBATCH -t 00:20:00
#SBATCH -c 2
#SBATCH --gres=gpu:a4000:1
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

#SBATCH -J alphafold
#SBATCH -p gpu-bf
#SBATCH --mem=16g
#SBATCH -t 01:00:00
#SBATCH -c 2
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH --gres=gpu
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

In [ ]:
import os
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Model import Model
from Bio.PDB.Structure import Structure
from Bio.PDB.Chain import Chain

# Directory containing the PDB files
pdb_dir = './mpnn/outputs/first_pass_superfold'

# Chain IDs to assign (from 'A' to 'Z', extendable if needed)
chain_ids = [chr(i) for i in range(ord('A'), ord('Z')+1)]

# Initialize PDB parser and PDBIO
parser = PDBParser(QUIET=True)
io = PDBIO()

# Iterate over PDB files in the directory
for pdb_filename in os.listdir(pdb_dir):
    if pdb_filename.endswith('.pdb'):
        pdb_path = os.path.join(pdb_dir, pdb_filename)
        print(f'Processing {pdb_path}')

        # Parse the structure
        structure = parser.get_structure('structure', pdb_path)

        # Assume only one model per structure
        model = structure[0]

        # Get chains and their lengths
        chains = list(model.get_chains())
        chain_lengths = []
        for chain in chains:
            residues = list(chain.get_residues())
            num_residues = len(residues)
            chain_lengths.append((chain, num_residues))

        # Sort chains by length (number of residues), from shortest to longest
        chain_lengths.sort(key=lambda x: x[1])

        # Create a new model to store chains with new IDs
        new_model = Model(model.id)

        # Reassign chain IDs and add to new model
        for idx, (chain, _) in enumerate(chain_lengths):
            if idx < len(chain_ids):
                new_chain_id = chain_ids[idx]
            else:
                # Extend chain IDs beyond 'Z' if necessary
                quotient, remainder = divmod(idx, len(chain_ids))
                new_chain_id = chain_ids[quotient - 1] + chain_ids[remainder]
            old_chain_id = chain.id

            # Create a new chain with the new ID
            new_chain = Chain(new_chain_id)

            # Copy residues from the old chain to the new chain
            for residue in chain.get_residues():
                new_chain.add(residue)

            # Add the new chain to the new model
            new_model.add(new_chain)
            print(f'Chain {old_chain_id} relabeled to {new_chain_id}')

        # Create a new structure with the new model
        new_structure = Structure(structure.id)
        new_structure.add(new_model)

        # Save the new structure back to a PDB file (overwrite original)
        io.set_structure(new_structure)
        io.save(pdb_path)
        print(f'Saved updated structure to {pdb_path}')

In [ ]:
import os
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser, Superimposer
import shutil

# Define directories
packed_dir = './mpnn/outputs/first_pass_superfold'
superfold_outputs_dir = './superfold/outputs'
rmsd_filtered_dir = './superfold/rmsd_filtered'

# Create the filtered directory if it doesn't exist
os.makedirs(rmsd_filtered_dir, exist_ok=True)

# Initialize PDB parser
parser = PDBParser(QUIET=True)

def calculate_rmsd(file1, file2):
    """Calculate RMSD between two PDB files using CA atoms."""
    try:
        structure1 = parser.get_structure('structure1', file1)
        structure2 = parser.get_structure('structure2', file2)

        # Extract CA atoms and order by residue number
        atoms1 = []
        atoms2 = []
        for model1, model2 in zip(structure1, structure2):
            for chain1, chain2 in zip(model1, model2):
                residues1 = [res for res in chain1 if res.has_id('CA')]
                residues2 = [res for res in chain2 if res.has_id('CA')]

                if len(residues1) != len(residues2):
                    print(f'Number of CA residues differ between {file1} and {file2}')
                    return None

                for res1, res2 in zip(residues1, residues2):
                    atoms1.append(res1['CA'])
                    atoms2.append(res2['CA'])

        if len(atoms1) != len(atoms2):
            print(f'Number of CA atoms differ between {file1} and {file2}')
            return None

        # Superimpose structures
        super_imposer = Superimposer()
        super_imposer.set_atoms(atoms1, atoms2)
        super_imposer.apply(structure2.get_atoms())

        return super_imposer.rms
    except Exception as e:
        print(f'Error calculating RMSD between {file1} and {file2}: {e}')
        return None

# Dataframe to store paths and RMSD values
data = []

# Iterate over files in the packed directory
for packed_pdb in os.listdir(packed_dir):
    if packed_pdb.endswith('.pdb'):
        packed_pdb_path = os.path.join(packed_dir, packed_pdb)
        # Loop over models 1 to 5
        for model_num in range(1, 6):
            # Construct corresponding file names in superfold directory
            superfold_pdb = packed_pdb.replace('.pdb', f'_model_{model_num}_ptm_seed_0_unrelaxed.pdb')
            print(f'Processing: {superfold_pdb}')
            superfold_json = superfold_pdb.replace('unrelaxed.pdb', 'prediction_results.json')
            
            superfold_pdb_path = os.path.join(superfold_outputs_dir, superfold_pdb)
            superfold_json_path = os.path.join(superfold_outputs_dir, superfold_json)
            
            # Ensure the corresponding superfold pdb and json files exist
            if os.path.exists(superfold_pdb_path) and os.path.exists(superfold_json_path):
                # Calculate RMSD
                rmsd = calculate_rmsd(packed_pdb_path, superfold_pdb_path)
                
                if rmsd is not None:
                    # Check RMSD threshold
                    if rmsd < 100:
                        # Copy pdb and json files to rmsd_filtered directory
                        shutil.copy(superfold_pdb_path, rmsd_filtered_dir)
                        shutil.copy(superfold_json_path, rmsd_filtered_dir)
                        
                        # Add to dataframe
                        data.append([packed_pdb_path, superfold_pdb_path, rmsd])
                else:
                    print(f'Skipping due to RMSD calculation error between {packed_pdb_path} and {superfold_pdb_path}')
            else:
                print(f'Missing files for {superfold_pdb_path} or {superfold_json_path}')

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data, columns=['Packed PDB Path', 'Superfold PDB Path', 'RMSD'])
csv_path = os.path.join(rmsd_filtered_dir, 'rmsd_filtered.csv')
df.to_csv(csv_path, index=False)

print(f'RMSD calculation and file filtering complete. Results saved to {csv_path}')

In [ ]:
import os
import json

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Loop through all files in the directory
for filename in os.listdir(directory):
    # Check if the file is a .json file
    if filename.endswith('prediction_results.json'):
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_plddt value
            mean_plddt = data.get('mean_plddt', 1.0)  # Default to 1.0 if not found
            
            # Print the mean_plddt value and the .json filename
            # print(f'{filename}: mean_plddt = {mean_plddt}')
            
            # Check if mean_plddt is less than 80
            if mean_plddt < 90:
                # Construct the corresponding .pdb filename
                pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
                # print(pdb_filename)
                pdb_path = os.path.join(directory, pdb_filename)
                # print(pdb_path)
                # Delete the .pdb file if it exists
                if os.path.exists(pdb_path):
                    os.remove(pdb_path)
                    print(f'Deleted {pdb_path}')
                else:
                    print(f'{pdb_path} does not exist')
                
                # Delete the .json file
                os.remove(json_path)
                print(f'Deleted {json_path}')

print('Processing complete.')

In [ ]:
import os
import json

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Loop through all files in the directory
for filename in os.listdir(directory):
    # Check if the file is a .json file
    if filename.endswith('prediction_results.json'):
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_pae value
            mean_pae = data.get('mean_pae', 999)  # Default to 999 if not found
            
            # Uncomment the following line if you want to print the mean_pae value and the .json filename
            # print(f'{filename}: mean_pae = {mean_pae}')
            
            # Check if mean_pae is less than 5
            if mean_pae > 5:
                # Construct the corresponding .pdb filename
                pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
                pdb_path = os.path.join(directory, pdb_filename)
                
                # Delete the .pdb file if it exists
                if os.path.exists(pdb_path):
                    os.remove(pdb_path)
                    print(f'Deleted {pdb_path}')
                else:
                    print(f'{pdb_path} does not exist')
                
                # Delete the .json file
                os.remove(json_path)
                print(f'Deleted {json_path}')

print('Processing complete.')

In [ ]:
# import os
# import json
# from collections import defaultdict

# # Define the directory containing the .json and .pdb files
# directory = './superfold/rmsd_filtered'

# # Initialize a dictionary to hold groups of files
# file_groups = defaultdict(list)

# # Function to extract the prefix based on the last instance of 'packed'
# def get_prefix(filename):
#     parts = filename.split('packed')
#     if len(parts) > 1:
#         return 'packed'.join(parts[:-1]) + 'packed'
#     return filename

# # Loop through all files in the directory
# for filename in os.listdir(directory):
#     if filename.endswith('prediction_results.json'):
#         prefix = get_prefix(filename)
#         json_path = os.path.join(directory, filename)
        
#         # Read the .json file
#         with open(json_path, 'r') as json_file:
#             data = json.load(json_file)
            
#             # Get the mean_plddt value
#             mean_plddt = data.get('mean_plddt', 0.0)  # Default to 0.0 if not found
            
#             # Append the file info to the group
#             file_groups[prefix].append((mean_plddt, json_path, filename))

# # Function to delete a file
# def delete_file(file_path):
#     if os.path.exists(file_path):
#         os.remove(file_path)
#         print(f'Deleted {file_path}')
#     else:
#         print(f'{file_path} does not exist')

# # Iterate through each group
# for prefix, files in file_groups.items():
#     # Sort the files by mean_plddt in descending order
#     files.sort(reverse=True, key=lambda x: x[0])
    
#     # Get the highest n files (modify 'n' to choose highest 1, 2, 3, etc.)
#     n = 1  # For highest 1 file per group, change to 3 for highest 3 per group
#     highest_files = files[:n]
    
#     # Files to keep
#     highest_files_set = set(f[1] for f in highest_files)
    
#     # Iterate through the files in the group
#     for _, json_path, filename in files:
#         if json_path not in highest_files_set:
#             # Delete the .json file
#             delete_file(json_path)
            
#             # Construct the corresponding .pdb filename and delete it
#             pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
#             pdb_path = os.path.join(directory, pdb_filename)
#             delete_file(pdb_path)

# print('Processing complete.')

import os
import json
from collections import defaultdict

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Initialize a dictionary to hold groups of files
file_groups = defaultdict(list)

# Function to extract the prefix based on the last instance of 'packed'
def get_prefix(filename):
    parts = filename.split('model')
    if len(parts) > 1:
        return 'model'.join(parts[:-1]) + 'model'
    return filename

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('prediction_results.json'):
        prefix = get_prefix(filename)
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_pae value; default to a high value if not found so it doesn't win
            mean_pae = data.get('mean_pae', 999)
            
            # Append the file info to the group
            file_groups[prefix].append((mean_pae, json_path, filename))

# Function to delete a file
def delete_file(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted {file_path}')
    else:
        print(f'{file_path} does not exist')

# Iterate through each group
for prefix, files in file_groups.items():
    # Sort the files by mean_pae in ascending order (lowest value first)
    files.sort(key=lambda x: x[0])
    
    # Get the lowest n files (modify 'n' to choose lowest 1, 2, 3, etc.)
    n = 1 # For the lowest 1 file per group; change to 3 for the lowest 3 per group
    lowest_files = files[:n]
    
    # Files to keep
    lowest_files_set = set(f[1] for f in lowest_files)
    
    # Iterate through the files in the group and delete those not selected
    for _, json_path, filename in files:
        if json_path not in lowest_files_set:
            # Delete the .json file
            delete_file(json_path)
            
            # Construct the corresponding .pdb filename and delete it
            pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
            pdb_path = os.path.join(directory, pdb_filename)
            delete_file(pdb_path)

print('Processing complete.')

In [ ]:
# #!/usr/bin/env python

# import os
# import glob
# from Bio import PDB
# from Bio.SeqRecord import SeqRecord
# from Bio import SeqIO
# from Bio.Seq import Seq

# # Directory containing PDB files
# # pdb_dir = './superfold/rmsd_filtered' 
# pdb_dir = './mpnn/outputs/packed'

# # Output FASTA file within pdb_dir
# fasta_file = os.path.join(pdb_dir, 'chainB.fasta')

# # The tail sequence to append after each chain B sequence
# # tail_seq = "KDRETLKEAIETIRELLEKILERFPGDKEILEKILDLIDELLDKTYKFCEEEKDRELLKETIETVRELLEKILEREPNDKTLLEKITKIIEKLLEKTYRFCEKEKDRELLKETIETVRELLEKIMERNPNDKELFKKILELVEKMLELTLKYAKKHKWSVEEFMELMSLIVELVIKLMEIVLKNN"
# tail_seq = ""

# # Initialize the PDB parser and list for sequence records
# parser = PDB.PDBParser(QUIET=True)
# records = []

# # Loop over all .pdb files in the directory
# for pdb_path in glob.glob(os.path.join(pdb_dir, '*.pdb')):
#     pdb_id = os.path.basename(pdb_path).split('.')[0]
#     try:
#         structure = parser.get_structure(pdb_id, pdb_path)
#     except Exception as e:
#         print(f"Error parsing {pdb_path}: {e}")
#         continue

#     # Assume using the first model
#     model = structure[0]

#     # Check if chain B exists
#     if 'B' not in model:
#         print(f"Chain B not found in {pdb_path}")
#         continue

#     chain_b = model['B']

#     # Build polypeptides from chain B (handles discontinuous segments)
#     ppb = PDB.PPBuilder()
#     polypeptides = ppb.build_peptides(chain_b)

#     # Concatenate sequences from all segments in chain B
#     seq_str = "".join([str(pp.get_sequence()) for pp in polypeptides])
#     if not seq_str:
#         print(f"No sequence extracted from chain B in {pdb_path}")
#         continue

#     # Append the tail sequence to the extracted chain B sequence
#     full_seq_str = seq_str + tail_seq

#     # Create a FASTA record with the full sequence (chain B sequence + tail)
#     record = SeqRecord(seq=Seq(full_seq_str), id=f"{pdb_id}_B", description="")
#     records.append(record)
#     print(f"Extracted chain B from {pdb_path} and appended tail sequence.")

# # Write all records to the FASTA file
# if records:
#     SeqIO.write(records, fasta_file, "fasta")
#     print(f"\nWrote {len(records)} record(s) to {fasta_file}")
# else:
#     print("No chain B sequences were extracted.")

#!/usr/bin/env python

import os
import glob
from Bio import PDB
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio.Seq import Seq

# Directory containing PDB files
pdb_dir = './superfold/rmsd_filtered'

# Output FASTA file within pdb_dir
fasta_file = os.path.join(pdb_dir, 'chainB.fasta')

# The head sequence to prepend before each chain B sequence
head_seq = "LKELVKEIIERTRELLEKILERYPENKELLEEILKLIEKMFKLVYEFCMKTKDKELFKETLERVRELLEKIAERNPENKELLEEILKLIETMLKKTKEFCLKFKLSVKEALELFKIIVEQLIEFFEIVLLNSDLKELAEELIKKLKEAIETAAEVVYELIKPESE"

# Initialize the PDB parser and list for sequence records
parser = PDB.PDBParser(QUIET=True)
records = []

# Loop over all .pdb files in the directory
for pdb_path in glob.glob(os.path.join(pdb_dir, '*.pdb')):
    pdb_id = os.path.basename(pdb_path).split('.')[0]
    try:
        structure = parser.get_structure(pdb_id, pdb_path)
    except Exception as e:
        print(f"Error parsing {pdb_path}: {e}")
        continue

    # Assume using the first model
    model = structure[0]

    # Check if chain B exists
    if 'B' not in model:
        print(f"Chain B not found in {pdb_path}")
        continue

    chain_b = model['B']

    # Build polypeptides from chain B (handles discontinuous segments)
    ppb = PDB.PPBuilder()
    polypeptides = ppb.build_peptides(chain_b)

    # Concatenate sequences from all segments in chain B
    seq_str = "".join([str(pp.get_sequence()) for pp in polypeptides])
    if not seq_str:
        print(f"No sequence extracted from chain B in {pdb_path}")
        continue

    # Prepend the head sequence to the extracted chain B sequence
    full_seq_str = head_seq + seq_str

    # Create a FASTA record with the full sequence (head sequence + chain B sequence)
    record = SeqRecord(seq=Seq(full_seq_str), id=f"{pdb_id}_B", description="")
    records.append(record)
    print(f"Extracted chain B from {pdb_path} and prepended head sequence.")

# Write all records to the FASTA file
if records:
    SeqIO.write(records, fasta_file, "fasta")
    print(f"\nWrote {len(records)} record(s) to {fasta_file}")
else:
    print("No chain B sequences were extracted.")

In [12]:
import os
import glob
from Bio.PDB import PDBParser, PPBuilder
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio.Seq import Seq

# Directory containing PDB files
glob_dir = './superfold/rmsd_filtered'
# Output FASTA file within pdb_dir
fasta_file = os.path.join(glob_dir, 'chainA_trimmed.fasta')
# The head sequence to prepend before each chain A sequence
head_seq = ""

# Initialize the PDB parser, peptide builder, and list for sequence records
parser = PDBParser(QUIET=True)
ppb = PPBuilder()
records = []

# Loop over all .pdb files in the directory
for pdb_path in glob.glob(os.path.join(glob_dir, '*.pdb')):
    pdb_id = os.path.basename(pdb_path).split('.')[0]
    try:
        structure = parser.get_structure(pdb_id, pdb_path)
    except Exception as e:
        print(f"Error parsing {pdb_path}: {e}")
        continue

    model = structure[0]
    if 'A' not in model:
        print(f"Chain A not found in {pdb_path}")
        continue
    chain_a = model['A']

    # Build polypeptides (handles discontinuous segments)
    peptides = ppb.build_peptides(chain_a)
    seq_str = "".join(str(pp.get_sequence()) for pp in peptides)

    if not seq_str:
        print(f"No sequence extracted from chain A in {pdb_path}")
        continue

    # Exclude first and last 34 residues
    if len(seq_str) <= 68:
        print(f"Sequence from chain A in {pdb_id} shorter than 68 aa; skipping trimming.")
        continue
    trimmed_seq = seq_str[:]

    # Prepend head sequence and create FASTA record
    full_seq = head_seq + trimmed_seq
    record = SeqRecord(Seq(full_seq), id=f"{pdb_id}_A", description="")
    records.append(record)
    print(f"Extracted trimmed chain A from {pdb_id} (residues 35 to {len(seq_str)-34}).")

# Write all records to the FASTA file
if records:
    SeqIO.write(records, fasta_file, 'fasta')
    print(f"\nWrote {len(records)} record(s) to {fasta_file}")
else:
    print("No trimmed chain A sequences were extracted.")

Extracted trimmed chain A from fiber_to_lhda_value1_107_value2_25_value3_80_ckpt_BFF_0_0_refined_2_packed_11_1_model_3_ptm_seed_0_unrelaxed (residues 35 to 398).
Extracted trimmed chain A from fiber_to_lhda_value1_103_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_11_1_model_3_ptm_seed_0_unrelaxed (residues 35 to 394).
Extracted trimmed chain A from fiber_to_lhda_value1_107_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_2_1_model_3_ptm_seed_0_unrelaxed (residues 35 to 398).
Extracted trimmed chain A from fiber_to_lhda_value1_103_value2_25_value3_80_ckpt_BFF_0_0_refined_1_packed_9_1_model_3_ptm_seed_0_unrelaxed (residues 35 to 394).
Extracted trimmed chain A from fiber_to_lhda_value1_109_value2_25_value3_80_ckpt_BFF_0_0_refined_0_packed_8_1_model_3_ptm_seed_0_unrelaxed (residues 35 to 400).

Wrote 5 record(s) to ./superfold/rmsd_filtered/chainA_trimmed.fasta


In [ ]:
# #!/usr/bin/env python

# import os
# import glob
# from Bio import PDB
# from Bio.SeqRecord import SeqRecord
# from Bio import SeqIO
# from Bio.Seq import Seq

# # Directory containing PDB files
# pdb_dir = './mpnn/outputs/packed'

# # Output FASTA file within pdb_dir
# fasta_file = os.path.join(pdb_dir, 'chainB.fasta')

# # Initialize the PDB parser and list for sequence records
# parser = PDB.PDBParser(QUIET=True)
# records = []

# # Loop over all .pdb files in the directory
# for pdb_path in glob.glob(os.path.join(pdb_dir, '*.pdb')):
#     pdb_id = os.path.basename(pdb_path).split('.')[0]
#     try:
#         structure = parser.get_structure(pdb_id, pdb_path)
#     except Exception as e:
#         print(f"Error parsing {pdb_path}: {e}")
#         continue

#     # Assume using the first model
#     model = structure[0]

#     # Check if chain B exists
#     if 'A' not in model:
#         print(f"Chain B not found in {pdb_path}")
#         continue

#     chain_b = model['A']

#     # Build polypeptides from chain B (handles discontinuous segments)
#     ppb = PDB.PPBuilder()
#     polypeptides = ppb.build_peptides(chain_b)

#     # Concatenate sequences from all segments in chain B
#     seq_str = "".join([str(pp.get_sequence()) for pp in polypeptides])
#     if not seq_str:
#         print(f"No sequence extracted from chain B in {pdb_path}")
#         continue

#     # Remove the last 88 residues
#     if len(seq_str) > 88:
#         trimmed_seq_str = seq_str[:-88]
#     else:
#         trimmed_seq_str = seq_str  # If sequence is shorter than 88 residues, keep it as is

#     # Create a FASTA record with the trimmed sequence
#     record = SeqRecord(seq=Seq(trimmed_seq_str), id=f"{pdb_id}_B", description="")
#     records.append(record)
#     print(f"Extracted and trimmed chain B from {pdb_path}.")

# # Write all records to the FASTA file
# if records:
#     SeqIO.write(records, fasta_file, "fasta")
#     print(f"\nWrote {len(records)} record(s) to {fasta_file}")
# else:
#     print("No chain B sequences were extracted.")

In [13]:
base_directory = "alphafold3"
create_directory_structure(base_directory)

import os
from Bio import SeqIO

def create_directory_structure(base_directory):
    """
    Creates a subdirectory named 'fasta_files' within the base_directory.
    Returns the full path to the created subdirectory.
    """
    output_directory = os.path.join(base_directory, "fasta_files")
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    return output_directory

def read_fasta(file_path):
    """
    Reads sequences from a FASTA file and returns a list of dictionaries,
    each containing the sequence name and its sequence string.
    """
    sequences = []
    for record in SeqIO.parse(file_path, "fasta"):
        sequences.append({'name': record.id, 'sequence': str(record.seq)})
    return sequences

def write_fasta_files(sequences, output_directory):
    """
    For each sequence, writes a separate FASTA file with the sequence repeated 3 times
    delimited by a space. The file is named after the sequence's name.
    """
    for seq in sequences:
        repeated_seq = "".join([seq['sequence']] * 4)
        fasta_content = f">{seq['name']}\n{repeated_seq}\n"
        fasta_file_path = os.path.join(output_directory, f"{seq['name']}.fasta")
        with open(fasta_file_path, 'w') as f:
            f.write(fasta_content)
        print(f"Created FASTA file: {fasta_file_path}")

if __name__ == "__main__":
    # Define base and output paths
    base_directory = "alphafold3"
    output_fasta_directory = create_directory_structure(base_directory)
    
    # Define the input FASTA file path
    input_fasta_path = './superfold/rmsd_filtered/chainA_trimmed.fasta'
    # input_fasta_path = './mpnn/outputs/packed/chainB.fasta'
    
    # Read sequences from the FASTA file
    sequences = read_fasta(input_fasta_path)
    
    # Write each sequence as a separate FASTA file with the sequence repeated 3 times
    write_fasta_files(sequences, output_fasta_directory)

Directory structure 'alphafold3/input' and 'alphafold3/output' created successfully.
Created FASTA file: alphafold3/fasta_files/107_25_80_refined_2_packed_11_1_0.fasta
Created FASTA file: alphafold3/fasta_files/107_25_80_refined_2_packed_11_1_1.fasta
Created FASTA file: alphafold3/fasta_files/103_25_80_refined_1_packed_11_1.fasta
Created FASTA file: alphafold3/fasta_files/107_25_80_refined_0_packed_2_1.fasta
Created FASTA file: alphafold3/fasta_files/103_25_80_refined_1_packed_9_1.fasta
Created FASTA file: alphafold3/fasta_files/109_25_80_refined_0_packed_8_1_0.fasta
Created FASTA file: alphafold3/fasta_files/109_25_80_refined_0_packed_8_1_1.fasta


In [14]:
import os
import glob

def main():
    # Define the absolute path for the tasks file
    tasks_file = "/home/pkwon2/cage/alphafold3/tasks"
    
    # Define the directory containing FASTA files
    fasta_dir = os.path.join("/home/pkwon2/cage/alphafold3", "fasta_files")
    
    # Find all FASTA files in the fasta_dir
    fasta_files = glob.glob(os.path.join(fasta_dir, "*.fasta"))
    
    # Write one command per FASTA file into the tasks file
    with open(tasks_file, "w") as file:
        for fasta_file in fasta_files:
            command = (
                "/software/containers/users/ikalvet/mlfold3/mlfold3_01.sif "
                "/net/software/lab/alphafold3/run_alphafold_custom.py "
                f"--fasta {fasta_file} "
                "--run_data_pipeline=false "
                "--output_dir ./outputs\n"
            )
            file.write(command)
    
    print(f"Tasks file '{tasks_file}' has been created with {len(fasta_files)} commands.")

if __name__ == "__main__":
    main()

Tasks file '/home/pkwon2/cage/alphafold3/tasks' has been created with 7 commands.


In [15]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J af3
#SBATCH -p gpu-bf
#SBATCH --mem=48g
#SBATCH -t 00:30:00
#SBATCH -c 12
#SBATCH --gres=gpu
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

SBATCH script 'alphafold3/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/pkwon2/cage/alphafold3
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [16]:
#!/usr/bin/env python3
import os
import glob
import shutil
from Bio.PDB import MMCIFParser, PDBIO

def convert_cif_to_pdb(cif_file, pdb_file):
    """
    Convert a CIF file to a PDB file using BioPython.
    """
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("structure", cif_file)
    io = PDBIO()
    io.set_structure(structure)
    io.save(pdb_file)

def main():
    # Define directories
    outputs_dir = "./alphafold3/outputs"
    pdb_dir = os.path.join(outputs_dir, "pdb")
    
    # Create pdb directory if it doesn't exist
    os.makedirs(pdb_dir, exist_ok=True)

    # Find all CIF files in subdirectories of outputs
    cif_pattern = os.path.join(outputs_dir, "*", "*.cif")
    cif_files = glob.glob(cif_pattern)
    
    # Find all summary_confidences.json files in subdirectories of outputs
    json_pattern = os.path.join(outputs_dir, "*", "*summary_confidences.json")
    json_files = glob.glob(json_pattern)

    # Process CIF files: convert to pdb and save in pdb_dir
    for cif in cif_files:
        base_name = os.path.basename(cif)
        pdb_filename = os.path.splitext(base_name)[0] + ".pdb"
        pdb_path = os.path.join(pdb_dir, pdb_filename)
        try:
            convert_cif_to_pdb(cif, pdb_path)
            print(f"Converted {cif} to {pdb_path}")
        except Exception as e:
            print(f"Failed to convert {cif}: {e}")

    # Process JSON files: copy them to pdb_dir instead of moving
    for json in json_files:
        dest = os.path.join(pdb_dir, os.path.basename(json))
        try:
            shutil.copy2(json, dest)
            print(f"Copied {json} to {dest}")
        except Exception as e:
            print(f"Failed to copy {json}: {e}")

if __name__ == "__main__":
    main()

Converted ./alphafold3/outputs/109_25_80_refined_0_packed_8_1_0/109_25_80_refined_0_packed_8_1_0_model.cif to ./alphafold3/outputs/pdb/109_25_80_refined_0_packed_8_1_0_model.pdb
Converted ./alphafold3/outputs/107_25_80_refined_0_packed_2_1/107_25_80_refined_0_packed_2_1_model.cif to ./alphafold3/outputs/pdb/107_25_80_refined_0_packed_2_1_model.pdb
Converted ./alphafold3/outputs/103_25_80_refined_1_packed_9_1/103_25_80_refined_1_packed_9_1_model.cif to ./alphafold3/outputs/pdb/103_25_80_refined_1_packed_9_1_model.pdb
Converted ./alphafold3/outputs/107_25_80_refined_2_packed_11_1_0/107_25_80_refined_2_packed_11_1_0_model.cif to ./alphafold3/outputs/pdb/107_25_80_refined_2_packed_11_1_0_model.pdb
Converted ./alphafold3/outputs/109_25_80_refined_0_packed_8_1_1/109_25_80_refined_0_packed_8_1_1_model.cif to ./alphafold3/outputs/pdb/109_25_80_refined_0_packed_8_1_1_model.pdb
Converted ./alphafold3/outputs/103_25_80_refined_1_packed_11_1/103_25_80_refined_1_packed_11_1_model.cif to ./alphafold

In [17]:
#!/usr/bin/env python3
import os
import glob
import csv
import json

def main():
    # Set the directory where the files are located.
    base_dir = "./alphafold3/outputs/pdb"
    
    # Search for all PDB files ending with '_model.pdb' in the base_dir.
    pdb_pattern = os.path.join(base_dir, "*_model.pdb")
    pdb_files = glob.glob(pdb_pattern)
    
    if not pdb_files:
        print(f"No PDB files found in {base_dir} matching '*_model.pdb'")
        return

    rows = []
    for pdb_file in pdb_files:
        # Derive the corresponding JSON filename by replacing '_model.pdb' with '_summary_confidences.json'
        json_filename = pdb_file.replace("_model.pdb", "_summary_confidences.json")
        if os.path.exists(json_filename):
            try:
                with open(json_filename, 'r') as f:
                    data = json.load(f)
                # Extract the desired metrics.
                fraction_disordered = data.get("fraction_disordered")
                has_clash = data.get("has_clash")
                iptm = data.get("iptm")
                ptm = data.get("ptm")
                ranking_score = data.get("ranking_score")
                
                rows.append({
                    "pdb_file": os.path.basename(pdb_file),
                    "fraction_disordered": fraction_disordered,
                    "has_clash": has_clash,
                    "iptm": iptm,
                    "ptm": ptm,
                    "ranking_score": ranking_score
                })
            except Exception as e:
                print(f"Error processing JSON file {json_filename}: {e}")
        else:
            print(f"Warning: JSON file {json_filename} not found for {pdb_file}")

    # Write the extracted metrics to a CSV file in the same directory.
    if rows:
        csv_path = os.path.join(base_dir, "metrics.csv")
        with open(csv_path, "w", newline='') as csvfile:
            fieldnames = ["pdb_file", "fraction_disordered", "has_clash", "iptm", "ptm", "ranking_score"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow(row)
        print(f"CSV file created successfully at: {csv_path}")
    else:
        print("No metrics were found to write to CSV.")

if __name__ == "__main__":
    main()

CSV file created successfully at: ./alphafold3/outputs/pdb/metrics.csv


In [ ]:
#!/usr/bin/env python3
"""
Filter AlphaFold-3 models by confidence metrics
----------------------------------------------
Deletes the .pdb and its *_summary_confidences.json if
  ipTM               < 0.80          OR
  pTM                < 0.50          OR
  ranking_score/conf < 0.75
Edit the IPTM_MIN, PTM_MIN and RANKING_MIN constants to change behaviour.
"""
import os
import csv

# ── thresholds ────────────────────────────────────────────────────────────────
IPTM_MIN   = 0.80
PTM_MIN    = 0.50
RANKING_MIN = 0.75               # applies to `ranking_score` or `ranking_confidence`
# ──────────────────────────────────────────────────────────────────────────────

def get_float(row, *keys, default=0.0):
    """Return the first numeric value found for any of the supplied keys."""
    for k in keys:
        val = row.get(k)
        if val is not None and val.strip() != "":
            try:
                return float(val)
            except ValueError:
                break
    return default

def should_delete(iptm, ptm, ranking):
    return iptm < IPTM_MIN or ptm < PTM_MIN or ranking < RANKING_MIN

def main():
    base_dir = "./alphafold3/outputs/pdb"
    csv_path = os.path.join(base_dir, "metrics.csv")

    if not os.path.exists(csv_path):
        print(f"[ERROR] CSV file not found: {csv_path}")
        return

    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            iptm    = get_float(row, "iptm")
            ptm     = get_float(row, "ptm")
            ranking = get_float(row, "ranking_confidence", "ranking_score")

            if ranking == 0 and "ranking_confidence" not in row and "ranking_score" not in row:
                print(f"[WARN] No ranking column found for row (skipped): {row}")
                continue

            if not should_delete(iptm, ptm, ranking):
                continue   # passes all cut-offs

            pdb_name = row.get("pdb_file", "").strip()
            if not pdb_name:
                print("[WARN] Row without 'pdb_file' (skipped).")
                continue

            pdb_path  = os.path.join(base_dir, pdb_name)
            json_name = (
                pdb_name.replace("_model.pdb", "_summary_confidences.json")
                if pdb_name.endswith("_model.pdb")
                else f"{os.path.splitext(pdb_name)[0]}_summary_confidences.json"
            )
            json_path = os.path.join(base_dir, json_name)

            for path in (pdb_path, json_path):
                if os.path.exists(path):
                    try:
                        os.remove(path)
                        print(f"[DEL] {os.path.relpath(path)}")
                    except Exception as e:
                        print(f"[ERROR] Could not delete {path}: {e}")
                else:
                    print(f"[INFO] File not found: {path}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os
import glob

# directories / paths
INPUT_DIR   = '/home/pkwon2/cage/alphafold3/outputs/pdb'
OUTPUT_FILE = '/home/pkwon2/cage/propagation/all_models.chimera_tasks'

# ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# gather all .pdb files
pdb_paths = sorted(glob.glob(os.path.join(INPUT_DIR, '*.pdb')))

with open(OUTPUT_FILE, 'w') as out:
    for pdb_path in pdb_paths:
        pdb_name = os.path.basename(pdb_path)
        # open the model three times
        for _ in range(3):
            out.write(f'open /Users/pkwon2/digs/cage/alphafold3/outputs/pdb/{pdb_name}\n')
        # open the reference three times
        for _ in range(3):
            out.write('open /Users/pkwon2/digs/cage/1_diffusion/input/tile_b.pdb\n')
        # matching / selection commands
        out.write('matchmaker #4/D to #1/A pairing ss\n')
        out.write('matchmaker #2/B to #4/E pairing ss\n')
        out.write('matchmaker #5/D to #2/A pairing ss\n')
        out.write('matchmaker #3/B to #5/E pairing ss\n')
        out.write('matchmaker #6/D to #3/A pairing ss\n')
        out.write('sel #4/D,E,F #5/D,E,F #6/D,E,F\n')
        out.write('del sel\n')
        # save and close
        out.write(f'save /Users/pkwon2/digs/cage/propagation/{pdb_name}\n')
        out.write('close\n\n')

print(f"Wrote Chimera tasks for {len(pdb_paths)} models to:\n  {OUTPUT_FILE}")

In [ ]:
#!/usr/bin/env python3
import os
import csv
import shutil
from glob import glob
import pymol2

# === Paths ===
af_pdb_dir    = '/home/pkwon2/cage/alphafold3/outputs/pdb'
ref_pdb       = '/home/pkwon2/cage/1_diffusion/input/tile_a.pdb'
filtered_dir  = '/home/pkwon2/cage/alphafold3/filtered'
out_csv       = os.path.join(filtered_dir, 'tile_a_rmsd.csv')

os.makedirs(filtered_dir, exist_ok=True)

with open(out_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['model_filename', 'rmsd_CA'])

    with pymol2.PyMOL() as pymol:
        cmd = pymol.cmd
        cmd.load(ref_pdb, 'ref')

        for model_fp in glob(os.path.join(af_pdb_dir, '*tile_a*.pdb')):
            model_name = os.path.basename(model_fp)
            base       = os.path.splitext(model_name)[0]

            # strip off everything after the last "_model" (so "<...>_model_2_ptm_seed_0_unrelaxed_b_model"
            # becomes "<...>_model_2_ptm_seed_0_unrelaxed_b")
            prefix = base.rsplit('_model', 1)[0]
            json_fp = os.path.join(af_pdb_dir, prefix + '_summary_confidences.json')

            # load & align
            cmd.load(model_fp, 'm1')
            rmsd = float(cmd.align('m1 and name CA', 'ref and name CA')[0])
            cmd.delete('m1')

            writer.writerow([model_name, f"{rmsd:.3f}"])

            if rmsd < 1.2:
                shutil.copy2(model_fp, filtered_dir)
                if os.path.isfile(json_fp):
                    shutil.copy2(json_fp, filtered_dir)
                else:
                    print(f"Warning: JSON not found for {model_name} → looked for {json_fp}")

        cmd.delete('ref')

print(f"Done. RMSD values written to {out_csv}")

In [ ]:
#!/usr/bin/env python3
import os
import csv
import shutil
from glob import glob
import pymol2

# === Paths ===
af_pdb_dir    = '/home/pkwon2/cage/alphafold3/outputs/pdb'
ref_pdb       = '/home/pkwon2/cage/1_diffusion/input/tile_b.pdb'
filtered_dir  = '/home/pkwon2/cage/alphafold3/filtered'
out_csv       = os.path.join(filtered_dir, 'tile_b_rmsd.csv')

os.makedirs(filtered_dir, exist_ok=True)

with open(out_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['model_filename', 'rmsd_CA'])

    with pymol2.PyMOL() as pymol:
        cmd = pymol.cmd
        cmd.load(ref_pdb, 'ref')

        for model_fp in glob(os.path.join(af_pdb_dir, '*tile_b*.pdb')):
            model_name = os.path.basename(model_fp)
            base       = os.path.splitext(model_name)[0]

            # strip off everything after the last "_model" (so "<...>_model_2_ptm_seed_0_unrelaxed_b_model"
            # becomes "<...>_model_2_ptm_seed_0_unrelaxed_b")
            prefix = base.rsplit('_model', 1)[0]
            json_fp = os.path.join(af_pdb_dir, prefix + '_summary_confidences.json')

            # load & align
            cmd.load(model_fp, 'm1')
            rmsd = float(cmd.align('m1 and name CA', 'ref and name CA')[0])
            cmd.delete('m1')

            writer.writerow([model_name, f"{rmsd:.3f}"])

            if rmsd < 1.2:
                shutil.copy2(model_fp, filtered_dir)
                if os.path.isfile(json_fp):
                    shutil.copy2(json_fp, filtered_dir)
                else:
                    print(f"Warning: JSON not found for {model_name} → looked for {json_fp}")

        cmd.delete('ref')

print(f"Done. RMSD values written to {out_csv}")

In [ ]:
base_directory = "mpnn"
create_directory_structure(base_directory)

In [ ]:
#!/usr/bin/env python3
import os

# directory containing your .pdb files
directory = '/home/pkwon2/cage/mpnn/input/'

for fname in os.listdir(directory):
    if not fname.endswith('.pdb'):
        continue

    if 'model' in fname:
        # split off 'model' and everything after
        # root = fname.split('model', 1)[0]
        root = fname.split('packed', 1)[0]
        # remove any trailing underscore(s)
        root = root.rstrip('_')
        new_name = root + '.pdb'

        src = os.path.join(directory, fname)
        dst = os.path.join(directory, new_name)

        # only rename if the target name is different
        if src != dst:
            os.rename(src, dst)
            print(f"Renamed:\n  {fname}\n→ {new_name}\n")

In [ ]:
#!/usr/bin/env python3
import os
from Bio import PDB

def find_non_charged_residues(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    residue_positions = []
    
    # Charged residues to conditionally exclude
    charged_residues = {'ARG', 'GLU', 'LYS'}
    # Numeric ranges in which to exclude those charged residues
    exclude_ranges = [
        range(3, 23),    # 3–22
        range(55, 75),   # 55–74
        range(295, 315)  # 295–314
    ]

    for model in structure:
        for chain in model:
            residues_list = list(chain.get_residues())
            # Exclude the last 37 residues from processing
            if len(residues_list) > 75:
                residues_to_process = residues_list[:-75]
            else:
                residues_to_process = []

            for residue in residues_to_process:
                resname = residue.get_resname()
                resnum  = residue.id[1]

                # If it's a charged residue AND its number is in any of the exclude ranges, skip it
                if resname in charged_residues and any(resnum in r for r in exclude_ranges):
                    continue

                # Otherwise, include it
                residue_positions.append(f"{chain.id}{resnum}")
                    
    return ' '.join(residue_positions)

def main():
    pdb_dir = './mpnn/input'
    pdb_dir = os.path.abspath(pdb_dir)
    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    base_directory = "mpnn"
    os.makedirs(base_directory, exist_ok=True)

    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--temperature 0.3 "
        "--redesigned_residues '{residues}' "
        "--homo_oligomer 1 "
        "--pdb_path {file_path} "
        "--out_folder outputs "
        "--batch_size 5 "
        "--number_of_batches 5 "
        "--bias_AA 'M:-10.0' "
        "--bias_AA 'R:-0.9' "
        "--pack_side_chains 1 "
        "--repack_everything 1"
    )

    tasks_file = os.path.join(base_directory, "tasks")
    with open(tasks_file, "w") as f:
        for pdb_file in os.listdir(pdb_dir):
            if not pdb_file.endswith('.pdb'):
                continue
            filepath = os.path.join(pdb_dir, pdb_file)
            residues = find_non_charged_residues(filepath)
            cmd = base_command.format(file_path=filepath, residues=residues)
            f.write(cmd + "\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [ ]:
# #!/usr/bin/env python3
# import os
# from Bio import PDB

# def find_non_charged_residues(file_path):
#     parser = PDB.PDBParser(QUIET=True)
#     structure = parser.get_structure('PDB_structure', file_path)
#     residue_positions = []
    
#     # Charged residues to exclude
#     charged_residues = {}

#     for model in structure:
#         for chain in model:
#             residues_list = list(chain.get_residues())
#             # Exclude the last 37 residues from processing
#             if len(residues_list) > 75:
#                 residues_to_process = residues_list[:-75]
#             else:
#                 residues_to_process = []

#             for residue in residues_to_process:
#                 resname = residue.get_resname()
#                 resnum  = residue.id[1]

#                 # Skip any charged residue (no special‐range check)
#                 if resname in charged_residues:
#                     continue

#                 # Otherwise, include it
#                 residue_positions.append(f"{chain.id}{resnum}")
                    
#     return ' '.join(residue_positions)

# def main():
#     pdb_dir = './mpnn/input'
#     pdb_dir = os.path.abspath(pdb_dir)
#     if not os.path.exists(pdb_dir):
#         print(f"Directory {pdb_dir} does not exist.")
#         return

#     base_directory = "mpnn"
#     os.makedirs(base_directory, exist_ok=True)

#     base_command = (
#         "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
#         "--temperature 0.3 "
#         "--redesigned_residues '{residues}' "
#         "--homo_oligomer 1 "
#         "--pdb_path {file_path} "
#         "--out_folder outputs "
#         "--batch_size 5 "
#         "--number_of_batches 5 "
#         "--bias_AA 'M:-10.0' "
#         "--bias_AA 'R:-0.9' "
#         "--pack_side_chains 1 "
#         "--repack_everything 1"
#     )

#     tasks_file = os.path.join(base_directory, "tasks")
#     with open(tasks_file, "w") as f:
#         for pdb_file in os.listdir(pdb_dir):
#             if not pdb_file.endswith('.pdb'):
#                 continue
#             filepath = os.path.join(pdb_dir, pdb_file)
#             residues = find_non_charged_residues(filepath)
#             cmd = base_command.format(file_path=filepath, residues=residues)
#             f.write(cmd + "\n")

#     print(f"Tasks file '{tasks_file}' has been created.")

# if __name__ == "__main__":
#     main()

In [ ]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J mpnn
#SBATCH -p cpu
#SBATCH --cpus-per-task=4
#SBATCH -t 04:00:00
#SBATCH --mem=16g
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

In [ ]:
import os
import glob
from Bio import PDB
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
from Bio.Seq import Seq

# Directory containing PDB files
pdb_dir = './mpnn/outputs/packed'

# Output FASTA file within pdb_dir
fasta_file = os.path.join(pdb_dir, 'chainB.fasta')

# The head sequence to prepend before each chain B sequence
head_seq = ""

# Initialize the PDB parser and list for sequence records
parser = PDB.PDBParser(QUIET=True)
records = []

# Loop over all .pdb files in the directory
for pdb_path in glob.glob(os.path.join(pdb_dir, '*.pdb')):
    pdb_id = os.path.basename(pdb_path).split('.')[0]
    try:
        structure = parser.get_structure(pdb_id, pdb_path)
    except Exception as e:
        print(f"Error parsing {pdb_path}: {e}")
        continue

    # Assume using the first model
    model = structure[0]

    # Check if chain B exists
    if 'B' not in model:
        print(f"Chain B not found in {pdb_path}")
        continue

    chain_b = model['B']

    # Build polypeptides from chain B (handles discontinuous segments)
    ppb = PDB.PPBuilder()
    polypeptides = ppb.build_peptides(chain_b)

    # Concatenate sequences from all segments in chain B
    seq_str = "".join([str(pp.get_sequence()) for pp in polypeptides])
    if not seq_str:
        print(f"No sequence extracted from chain B in {pdb_path}")
        continue

    # Prepend the head sequence to the extracted chain B sequence
    full_seq_str = head_seq + seq_str

    # Create a FASTA record with the full sequence (head sequence + chain B sequence)
    record = SeqRecord(seq=Seq(full_seq_str), id=f"{pdb_id}_B", description="")
    records.append(record)
    print(f"Extracted chain B from {pdb_path} and prepended head sequence.")

# Write all records to the FASTA file
if records:
    SeqIO.write(records, fasta_file, "fasta")
    print(f"\nWrote {len(records)} record(s) to {fasta_file}")
else:
    print("No chain B sequences were extracted.")

In [ ]:
base_directory = "alphafold3"
create_directory_structure(base_directory)

import os
from Bio import SeqIO

def create_directory_structure(base_directory):
    """
    Creates a subdirectory named 'fasta_files' within the base_directory.
    Returns the full path to the created subdirectory.
    """
    output_directory = os.path.join(base_directory, "fasta_files")
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    return output_directory

def read_fasta(file_path):
    """
    Reads sequences from a FASTA file and returns a list of dictionaries,
    each containing the sequence name and its sequence string.
    """
    sequences = []
    for record in SeqIO.parse(file_path, "fasta"):
        sequences.append({'name': record.id, 'sequence': str(record.seq)})
    return sequences

def write_fasta_files(sequences, output_directory):
    """
    For each sequence, writes a separate FASTA file with the sequence repeated 3 times
    delimited by a space. The file is named after the sequence's name.
    """
    for seq in sequences:
        repeated_seq = " ".join([seq['sequence']] * 3)
        fasta_content = f">{seq['name']}\n{repeated_seq}\n"
        fasta_file_path = os.path.join(output_directory, f"{seq['name']}.fasta")
        with open(fasta_file_path, 'w') as f:
            f.write(fasta_content)
        print(f"Created FASTA file: {fasta_file_path}")

if __name__ == "__main__":
    # Define base and output paths
    base_directory = "alphafold3"
    output_fasta_directory = create_directory_structure(base_directory)
    
    # Define the input FASTA file path
    # input_fasta_path = './superfold/rmsd_filtered/chainB.fasta'
    input_fasta_path = './mpnn/outputs/packed/chainB.fasta'
    
    # Read sequences from the FASTA file
    sequences = read_fasta(input_fasta_path)
    
    # Write each sequence as a separate FASTA file with the sequence repeated 3 times
    write_fasta_files(sequences, output_fasta_directory)

In [ ]:
import os
import glob

def main():
    # Define the absolute path for the tasks file
    tasks_file = "/home/pkwon2/cage/alphafold3/tasks"
    
    # Define the directory containing FASTA files
    fasta_dir = os.path.join("/home/pkwon2/cage/alphafold3", "fasta_files")
    
    # Find all FASTA files in the fasta_dir
    fasta_files = glob.glob(os.path.join(fasta_dir, "*.fasta"))
    
    # Write one command per FASTA file into the tasks file
    with open(tasks_file, "w") as file:
        for fasta_file in fasta_files:
            command = (
                "/software/containers/users/ikalvet/mlfold3/mlfold3_01.sif "
                "/net/software/lab/alphafold3/run_alphafold_custom.py "
                f"--fasta {fasta_file} "
                "--run_data_pipeline=false "
                "--output_dir ./outputs\n"
            )
            file.write(command)
    
    print(f"Tasks file '{tasks_file}' has been created with {len(fasta_files)} commands.")

if __name__ == "__main__":
    main()

# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J af3
#SBATCH -p gpu-bf
#SBATCH --mem=48g
#SBATCH -t 00:30:00
#SBATCH -c 12
#SBATCH --gres=gpu
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

In [ ]:
#!/usr/bin/env python3
import os
import glob
import shutil
from Bio.PDB import MMCIFParser, PDBIO

def convert_cif_to_pdb(cif_file, pdb_file):
    """
    Convert a CIF file to a PDB file using BioPython.
    """
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("structure", cif_file)
    io = PDBIO()
    io.set_structure(structure)
    io.save(pdb_file)

def main():
    # Define directories
    outputs_dir = "./alphafold3/outputs"
    pdb_dir = os.path.join(outputs_dir, "pdb")
    
    # Create pdb directory if it doesn't exist
    os.makedirs(pdb_dir, exist_ok=True)

    # Find all CIF files in subdirectories of outputs
    cif_pattern = os.path.join(outputs_dir, "*", "*.cif")
    cif_files = glob.glob(cif_pattern)
    
    # Find all summary_confidences.json files in subdirectories of outputs
    json_pattern = os.path.join(outputs_dir, "*", "*summary_confidences.json")
    json_files = glob.glob(json_pattern)

    # Process CIF files: convert to pdb and save in pdb_dir
    for cif in cif_files:
        base_name = os.path.basename(cif)
        pdb_filename = os.path.splitext(base_name)[0] + ".pdb"
        pdb_path = os.path.join(pdb_dir, pdb_filename)
        try:
            convert_cif_to_pdb(cif, pdb_path)
            print(f"Converted {cif} to {pdb_path}")
        except Exception as e:
            print(f"Failed to convert {cif}: {e}")

    # Process JSON files: copy them to pdb_dir instead of moving
    for json in json_files:
        dest = os.path.join(pdb_dir, os.path.basename(json))
        try:
            shutil.copy2(json, dest)
            print(f"Copied {json} to {dest}")
        except Exception as e:
            print(f"Failed to copy {json}: {e}")

if __name__ == "__main__":
    main()

#!/usr/bin/env python3
import os
import glob
import csv
import json

def main():
    # Set the directory where the files are located.
    base_dir = "./alphafold3/outputs/pdb"
    
    # Search for all PDB files ending with '_model.pdb' in the base_dir.
    pdb_pattern = os.path.join(base_dir, "*_model.pdb")
    pdb_files = glob.glob(pdb_pattern)
    
    if not pdb_files:
        print(f"No PDB files found in {base_dir} matching '*_model.pdb'")
        return

    rows = []
    for pdb_file in pdb_files:
        # Derive the corresponding JSON filename by replacing '_model.pdb' with '_summary_confidences.json'
        json_filename = pdb_file.replace("_model.pdb", "_summary_confidences.json")
        if os.path.exists(json_filename):
            try:
                with open(json_filename, 'r') as f:
                    data = json.load(f)
                # Extract the desired metrics.
                fraction_disordered = data.get("fraction_disordered")
                has_clash = data.get("has_clash")
                iptm = data.get("iptm")
                ptm = data.get("ptm")
                ranking_score = data.get("ranking_score")
                
                rows.append({
                    "pdb_file": os.path.basename(pdb_file),
                    "fraction_disordered": fraction_disordered,
                    "has_clash": has_clash,
                    "iptm": iptm,
                    "ptm": ptm,
                    "ranking_score": ranking_score
                })
            except Exception as e:
                print(f"Error processing JSON file {json_filename}: {e}")
        else:
            print(f"Warning: JSON file {json_filename} not found for {pdb_file}")

    # Write the extracted metrics to a CSV file in the same directory.
    if rows:
        csv_path = os.path.join(base_dir, "metrics.csv")
        with open(csv_path, "w", newline='') as csvfile:
            fieldnames = ["pdb_file", "fraction_disordered", "has_clash", "iptm", "ptm", "ranking_score"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow(row)
        print(f"CSV file created successfully at: {csv_path}")
    else:
        print("No metrics were found to write to CSV.")

if __name__ == "__main__":
    main()
    
#!/usr/bin/env python3
"""
Filter AlphaFold-3 models by confidence metrics
----------------------------------------------
Deletes the .pdb and its *_summary_confidences.json if
  ipTM               < 0.80          OR
  pTM                < 0.50          OR
  ranking_score/conf < 0.75
Edit the IPTM_MIN, PTM_MIN and RANKING_MIN constants to change behaviour.
"""
import os
import csv

# ── thresholds ────────────────────────────────────────────────────────────────
IPTM_MIN   = 0.80
PTM_MIN    = 0.50
RANKING_MIN = 0.75               # applies to `ranking_score` or `ranking_confidence`
# ──────────────────────────────────────────────────────────────────────────────

def get_float(row, *keys, default=0.0):
    """Return the first numeric value found for any of the supplied keys."""
    for k in keys:
        val = row.get(k)
        if val is not None and val.strip() != "":
            try:
                return float(val)
            except ValueError:
                break
    return default

def should_delete(iptm, ptm, ranking):
    return iptm < IPTM_MIN or ptm < PTM_MIN or ranking < RANKING_MIN

def main():
    base_dir = "./alphafold3/outputs/pdb"
    csv_path = os.path.join(base_dir, "metrics.csv")

    if not os.path.exists(csv_path):
        print(f"[ERROR] CSV file not found: {csv_path}")
        return

    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            iptm    = get_float(row, "iptm")
            ptm     = get_float(row, "ptm")
            ranking = get_float(row, "ranking_confidence", "ranking_score")

            if ranking == 0 and "ranking_confidence" not in row and "ranking_score" not in row:
                print(f"[WARN] No ranking column found for row (skipped): {row}")
                continue

            if not should_delete(iptm, ptm, ranking):
                continue   # passes all cut-offs

            pdb_name = row.get("pdb_file", "").strip()
            if not pdb_name:
                print("[WARN] Row without 'pdb_file' (skipped).")
                continue

            pdb_path  = os.path.join(base_dir, pdb_name)
            json_name = (
                pdb_name.replace("_model.pdb", "_summary_confidences.json")
                if pdb_name.endswith("_model.pdb")
                else f"{os.path.splitext(pdb_name)[0]}_summary_confidences.json"
            )
            json_path = os.path.join(base_dir, json_name)

            for path in (pdb_path, json_path):
                if os.path.exists(path):
                    try:
                        os.remove(path)
                        print(f"[DEL] {os.path.relpath(path)}")
                    except Exception as e:
                        print(f"[ERROR] Could not delete {path}: {e}")
                else:
                    print(f"[INFO] File not found: {path}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
import os
import csv
import shutil
from glob import glob
import pymol2

# === Paths ===
af_pdb_dir    = '/home/pkwon2/cage/alphafold3/outputs/pdb'
ref_pdb       = '/home/pkwon2/cage/1_diffusion/input/partial_tile_a.pdb'
filtered_dir  = '/home/pkwon2/cage/alphafold3/filtered'
out_csv       = os.path.join(filtered_dir, 'tile_a_rmsd.csv')

os.makedirs(filtered_dir, exist_ok=True)

with open(out_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['model_filename', 'rmsd_CA'])

    with pymol2.PyMOL() as pymol:
        cmd = pymol.cmd
        cmd.load(ref_pdb, 'ref')

        for model_fp in glob(os.path.join(af_pdb_dir, '*tile_a*.pdb')):
            model_name = os.path.basename(model_fp)
            base       = os.path.splitext(model_name)[0]

            # strip off everything after the last "_model" (so "<...>_model_2_ptm_seed_0_unrelaxed_b_model"
            # becomes "<...>_model_2_ptm_seed_0_unrelaxed_b")
            prefix = base.rsplit('_model', 1)[0]
            json_fp = os.path.join(af_pdb_dir, prefix + '_summary_confidences.json')

            # load & align
            cmd.load(model_fp, 'm1')
            rmsd = float(cmd.align('m1 and name CA', 'ref and name CA')[0])
            cmd.delete('m1')

            writer.writerow([model_name, f"{rmsd:.3f}"])

            if rmsd < 1.0:
                shutil.copy2(model_fp, filtered_dir)
                if os.path.isfile(json_fp):
                    shutil.copy2(json_fp, filtered_dir)
                else:
                    print(f"Warning: JSON not found for {model_name} → looked for {json_fp}")

        cmd.delete('ref')

print(f"Done. RMSD values written to {out_csv}")

In [ ]:
#!/usr/bin/env python3
import os
import glob

# directories / paths
INPUT_DIR   = '/home/pkwon2/cage/alphafold3/filtered'
OUTPUT_FILE = '/home/pkwon2/cage/propagation/chimera_tasks.txt'

# ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# gather all .pdb files
pdb_paths = sorted(glob.glob(os.path.join(INPUT_DIR, '*.pdb')))

with open(OUTPUT_FILE, 'w') as out:
    for pdb_path in pdb_paths:
        pdb_name = os.path.basename(pdb_path)
        # open the model three times
        for _ in range(3):
            out.write(f'open /Users/pkwon2/digs/cage/alphafold3/outputs/pdb/{pdb_name}\n')
        # open the reference three times
        for _ in range(3):
            out.write('open /Users/pkwon2/digs/cage/1_diffusion/input/tile_b.pdb\n')
        # matching / selection commands
        out.write('matchmaker #4/D to #1/A pairing ss\n')
        out.write('matchmaker #2/B to #4/E pairing ss\n')
        out.write('matchmaker #5/D to #2/A pairing ss\n')
        out.write('matchmaker #3/B to #5/E pairing ss\n')
        out.write('matchmaker #6/D to #3/A pairing ss\n')
        out.write('sel #4/D,E,F #5/D,E,F #6/D,E,F\n')
        out.write('del sel\n')
        # save and close
        out.write(f'save /Users/pkwon2/digs/cage/propagation/{pdb_name}\n')
        out.write('close\n\n')

print(f"Wrote Chimera tasks for {len(pdb_paths)} models to:\n  {OUTPUT_FILE}")

In [ ]:
#!/usr/bin/env python3
import os
import csv
import shutil
from glob import glob
import pymol2

# === Paths ===
af_pdb_dir    = '/home/pkwon2/cage/alphafold3/outputs/pdb'
ref_pdb       = '/home/pkwon2/cage/1_diffusion/input/tile_b.pdb'
filtered_dir  = '/home/pkwon2/cage/alphafold3/filtered'
out_csv       = os.path.join(filtered_dir, 'tile_b_rmsd.csv')

os.makedirs(filtered_dir, exist_ok=True)

with open(out_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['model_filename', 'rmsd_CA'])

    with pymol2.PyMOL() as pymol:
        cmd = pymol.cmd
        cmd.load(ref_pdb, 'ref')

        for model_fp in glob(os.path.join(af_pdb_dir, '*tile_b*.pdb')):
            model_name = os.path.basename(model_fp)
            base       = os.path.splitext(model_name)[0]

            # strip off everything after the last "_model"
            prefix = base.rsplit('_model', 1)[0]
            json_fp = os.path.join(af_pdb_dir, prefix + '_summary_confidences.json')

            # load model
            cmd.load(model_fp, 'm1')

            # run cealign on CA atoms
            ce_res = cmd.cealign('m1 and name CA', 'ref and name CA')

            # extract RMSD, depending on return type
            if isinstance(ce_res, (list, tuple)) and len(ce_res) >= 1:
                rmsd = float(ce_res[0])
            elif isinstance(ce_res, dict) and 'RMSD' in ce_res:
                rmsd = float(ce_res['RMSD'])
            else:
                # if neither format matches, set RMSD = 0.0 (or handle as you like)
                rmsd = 0.0

            # delete the model object before writing out
            cmd.delete('m1')

            writer.writerow([model_name, f"{rmsd:.3f}"])

            if rmsd < 0.6:
                shutil.copy2(model_fp, filtered_dir)
                if os.path.isfile(json_fp):
                    shutil.copy2(json_fp, filtered_dir)
                else:
                    print(f"Warning: JSON not found for {model_name} → looked for {json_fp}")

        cmd.delete('ref')

print(f"Done. RMSD values written to {out_csv}")

In [ ]:
# #!/usr/bin/env python3
# import os
# import json
# import random
# from collections import defaultdict

# # Directory where your files live
# BASE_DIR = "/home/pkwon2/cage/alphafold3/filtered"

# def get_group_key(filename):
#     # take everything before the first '_packed'
#     return filename.split("_packed", 1)[0]

# def find_pairs(directory):
#     """
#     Returns a list of tuples: (group_key, stem, pdb_path, json_path, metrics_dict)
#     """
#     pairs = []
#     for fname in os.listdir(directory):
#         if not fname.endswith("_summary_confidences.json"):
#             continue
#         json_path = os.path.join(directory, fname)
#         # derive the model filename
#         stem = fname[:-len("_summary_confidences.json")]
#         pdb_fname = stem + "_model.pdb"
#         pdb_path = os.path.join(directory, pdb_fname)
#         if not os.path.isfile(pdb_path):
#             # no matching pdb—skip
#             continue

#         # load metrics
#         with open(json_path) as f:
#             data = json.load(f)
#         metrics = {
#             "ranking_score": data.get("ranking_score", 0),
#             "iptm": data.get("iptm", 0),
#             "ptm": data.get("ptm", 0),
#         }
#         group = get_group_key(stem)
#         pairs.append((group, stem, pdb_path, json_path, metrics))
#     return pairs

# def choose_best(pairs_in_group):
#     # shuffle to randomize tie-break order
#     random.shuffle(pairs_in_group)
#     # sort by ranking_score, iptm, ptm (all descending)
#     sorted_list = sorted(
#         pairs_in_group,
#         key=lambda x: (
#             x[4]["ranking_score"],
#             x[4]["iptm"],
#             x[4]["ptm"]
#         ),
#         reverse=True
#     )
#     # first element is best
#     return sorted_list[0]

# def main():
#     all_pairs = find_pairs(BASE_DIR)

#     # group them
#     groups = defaultdict(list)
#     for group, stem, pdb, jsn, metrics in all_pairs:
#         groups[group].append((group, stem, pdb, jsn, metrics))

#     # for each group, pick best and delete the rest
#     for group_key, items in groups.items():
#         if len(items) <= 1:
#             continue  # nothing to delete

#         best = choose_best(items)
#         best_stem = best[1]

#         for grp, stem, pdb_path, json_path, metrics in items:
#             if stem == best_stem:
#                 print(f"[KEEP] {os.path.basename(pdb_path)} & {os.path.basename(json_path)}")
#             else:
#                 print(f"[DEL ] {os.path.basename(pdb_path)} & {os.path.basename(json_path)}")
#                 try:
#                     os.remove(pdb_path)
#                     os.remove(json_path)
#                 except Exception as e:
#                     print(f"  ERROR deleting {stem!r}: {e}")

# if __name__ == "__main__":
#     main()

In [19]:
#!/usr/bin/env python3
import os
import glob
from Bio.PDB import PDBParser, PDBIO, Select

class ChainASelect(Select):
    """Select only chain A."""
    def accept_chain(self, chain):
        return chain.id == 'A'

def main():
    order_dir = "./order"
    cowboy_dir = "./cowboy"
    os.makedirs(cowboy_dir, exist_ok=True)

    # Get all PDB files in the order directory
    pdb_files = glob.glob(os.path.join(order_dir, "*.pdb"))
    parser = PDBParser(QUIET=True)
    
    for pdb_file in pdb_files:
        structure = parser.get_structure(os.path.basename(pdb_file), pdb_file)
        io = PDBIO()
        io.set_structure(structure)
        output_path = os.path.join(cowboy_dir, os.path.basename(pdb_file))
        try:
            io.save(output_path, select=ChainASelect())
            print(f"Copied chain A from {pdb_file} to {output_path}")
        except Exception as e:
            print(f"Failed to copy chain A from {pdb_file}: {e}")

if __name__ == "__main__":
    main()

Copied chain A from ./order/107_25_80_refined_2_packed_11_1_0_model_packed_3_1_model.pdb to ./cowboy/107_25_80_refined_2_packed_11_1_0_model_packed_3_1_model.pdb
Copied chain A from ./order/103_25_80_refined_1_packed_11_1_model_packed_2_1_model.pdb to ./cowboy/103_25_80_refined_1_packed_11_1_model_packed_2_1_model.pdb
Copied chain A from ./order/103_25_80_refined_1_packed_9_1_model_packed_2_1_model.pdb to ./cowboy/103_25_80_refined_1_packed_9_1_model_packed_2_1_model.pdb
Copied chain A from ./order/partial_tile_a_packed_4_1_b_model.pdb to ./cowboy/partial_tile_a_packed_4_1_b_model.pdb
Copied chain A from ./order/partial_tile_a_packed_18_1_b_model.pdb to ./cowboy/partial_tile_a_packed_18_1_b_model.pdb
Copied chain A from ./order/109_25_80_refined_0_packed_8_1_1_model_packed_3_1_model.pdb to ./cowboy/109_25_80_refined_0_packed_8_1_1_model_packed_3_1_model.pdb
Copied chain A from ./order/partial_tile_a_packed_23_1_b_model.pdb to ./cowboy/partial_tile_a_packed_23_1_b_model.pdb
Copied chain

/net/software/lab/johnbercow/JohnBercow.py --verbose --order_pdbs ./ --order_name 2025_07_15_fibersNtile_a --design_prefix LM --design_id 1 --gg_vector LM0627 --species e_coli --no_layout

In [ ]:
import pandas as pd
import os

# Define the input and output directories
input_dir = './cowboy/'
output_dir = './modified_csv_files/'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Specify the input file names
excel_file = os.path.join(input_dir, '2025_05_16_2025_05_16_array_e_coli_BsaI.xlsx')  # Replace with your actual Excel file name
csv_file = os.path.join(input_dir, '2025_05_16_2025_05_16_array_e_coli_BsaI_df.csv')       # Replace with your actual CSV file name

# Read the Excel file
df_excel = pd.read_excel(excel_file, engine='openpyxl')

# Read the CSV file
df_csv = pd.read_csv(csv_file)

# Check if necessary columns exist in the Excel file
if 'Well Position' not in df_excel.columns or 'Name' not in df_excel.columns:
    print(f"Required columns 'Well Position' and 'Name' are missing in {excel_file}.")
    exit()

# Check if necessary columns exist in the CSV file
if 'Name' not in df_csv.columns or 'ORF_from_LM0627' not in df_csv.columns:
    print(f"Required columns 'Name' and 'ORF_from_LM0627' are missing in {csv_file}.")
    exit()

# Remove the first 'atg' from each entry in 'ORF_from_LM0627' and rename the column to 'Sequence'
df_csv['Sequence'] = df_csv['ORF_from_LM0627'].str.replace('^atg', '', case=False, regex=True)

# Keep only the 'Name' and 'Sequence' columns from the CSV file
df_csv = df_csv[['Name', 'Sequence']]

# Keep only the 'Well Position' and 'Name' columns from the Excel file
df_excel = df_excel[['Well Position', 'Name']]

# Merge the DataFrames on the 'Name' column
merged_df = pd.merge(df_excel, df_csv, on='Name', how='inner')

# Define the output file path
output_file = os.path.join(output_dir, 'merged_output.xlsx')

# Save the merged DataFrame to an Excel file
merged_df.to_excel(output_file, index=False)

print(f"Merged data saved to {output_file}")